In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗RTF
# ║  FME REVISION — SEQUENTIAL SESSIONS WITH PERSISTENT CACHE              ║
# ║  Each phase runs in its own session, cache accumulates across all      ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  HOW TO USE (Kaggle):                                                   ║
# ║   1. New Notebook → GPU P100 → Internet ON.                             ║
# ║   2. Paste PART 1 into Cell 1, PART 2 into Cell 2.                     ║
# ║   3. Run Cell 1 ONCE to set up infrastructure (always).                ║
# ║   4. Run Cell 2 to execute ONE phase per session.                       ║
# ║      Set CURRENT_PHASE to whichever phase you want to run.             ║
# ║   5. Cache is saved to /kaggle/working/fme_cache/ after every job.     ║
# ║   6. Each new session: run Cell 1, change CURRENT_PHASE, run Cell 2.   ║
# ║   7. When all phases done → final summary prints automatically.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ============================ USER CONFIG ==================================
SMOKE = False                 # True => tiny local code-test mode (no downloads)
TIME_BUDGET_MINUTES = 999999  # hard stop per session

# ══════════════════════════════════════════════════════════════════
# SET THIS BEFORE EACH SESSION — which phase to run in THIS session
# ══════════════════════════════════════════════════════════════════
CURRENT_PHASE = 'B'   # Change to: 'D','B','C','E','F','A','G','H'
# ══════════════════════════════════════════════════════════════════

# Phase descriptions (for reference):
#   D = unified RTF measurement, both protocols        (R1-6, R2-11, R2-12)  ~10 min
#   B = Hurst regularizer ON/OFF, 5 seeds              (R2-7, R1-5)          ~40 min
#   C = Table-2 quick config, 8 seeds                  (R2-14, R2-9, R2-3)   ~2 h
#   E = FME on REAL corpora CREMA-D + RAVDESS          (R1-1, R2-1)          ~3 h
#   F = SSL baselines FINE-TUNED on same real data     (R1-2, R2-8, R2-11)   ~8 h
#   A = published ablation suite, 8 seeds (128 jobs)   (R2-2/5/6/13/14/15)   ~6 h
#   G = long-form 20s/60s, scales 1/4/5, 3 seeds       (R2-4)                ~2 h
#   H = CTC lexical decoding on LibriSpeech clean      (R1-4, R2-3)          ~4 h

PHASES_ENABLED = ['D', 'B', 'C', 'E', 'F', 'A', 'G', 'H']  # all valid phases
SEEDS_A = [42, 43, 44, 45, 46, 47, 48, 49]
SEEDS_B = [42, 43, 44, 45, 46]
SEEDS_C = [42, 43, 44, 45, 46, 47, 48, 49]
SEEDS_E = [42, 43, 44]
SEEDS_F = [42, 43]
SEEDS_G = [42, 43, 44]
REAL_CORPORA = ['cremad', 'ravdess']
SSL_MODELS = {
    'wav2vec2': 'facebook/wav2vec2-base',
    'hubert':   'facebook/hubert-base-ls960',
    'wavlm':    'microsoft/wavlm-base',
}
TRY_EMOTION2VEC = True
CTC_EPOCHS = 3

# ── Cache lives in /kaggle/working so it persists across notebook sessions ──
CACHE_DIR_NAME = '/kaggle/working/fme_cache'
# ========================== END USER CONFIG ================================

import gc, os, sys, time, json, math, random, subprocess, traceback, warnings
import urllib.request, zipfile, tarfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

def _pip_install(pkgs):
    for p in pkgs:
        try:
            __import__(p.replace('-', '_').split('[')[0])
        except Exception:
            try:
                subprocess.check_call(
                    [sys.executable, '-m', 'pip', 'install', '-q', p],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception as e:
                print(f"  [pip] could not install {p}: {e}")

_pip_install(['torchaudio', 'soundfile', 'einops', 'scikit-learn',
              'matplotlib', 'seaborn', 'tqdm', 'pandas', 'scipy'])
import scipy
from scipy import stats as st
import sklearn

# ── Persistent cache directory ──────────────────────────────────────────────
CACHE_DIR = Path(CACHE_DIR_NAME)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / 'mega_checkpoint.json'

# ── Per-phase status file (tracks which phases have been fully completed) ───
PHASE_STATUS_FILE = CACHE_DIR / 'phase_status.json'

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print("=" * 74)
print("  FME ALL-EXPERIMENTS — SEQUENTIAL SESSIONS")
print(f"  device={DEVICE}" + (f" ({torch.cuda.get_device_name(0)})"
      if DEVICE.type == 'cuda' else ""))
print(f"  torch={torch.__version__}  SMOKE={SMOKE}  budget={TIME_BUDGET_MINUTES} min")
print(f"  CURRENT_PHASE={CURRENT_PHASE}")
print(f"  cache={CACHE_FILE}")
print("=" * 74)

# ── Validate phase selection ─────────────────────────────────────────────────
if CURRENT_PHASE not in PHASES_ENABLED:
    raise ValueError(
        f"CURRENT_PHASE='{CURRENT_PHASE}' is not in PHASES_ENABLED={PHASES_ENABLED}. "
        f"Please set CURRENT_PHASE to one of: {PHASES_ENABLED}")

def set_run_seed(seed, run_idx):
    s = int(seed) * 1000 + int(run_idx)
    random.seed(s)
    np.random.seed(s % (2**32))
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

# ── Load cumulative cache (all phases from all previous sessions) ────────────
CACHE = {}
if CACHE_FILE.exists():
    try:
        CACHE = json.load(open(CACHE_FILE))
        print(f"[resume] cache loaded: {len(CACHE)} finished jobs "
              f"from previous sessions")
    except Exception as e:
        print(f"[resume] cache unreadable ({e}); starting fresh")
        CACHE = {}

# ── Load phase status ────────────────────────────────────────────────────────
PHASE_STATUS = {}
if PHASE_STATUS_FILE.exists():
    try:
        PHASE_STATUS = json.load(open(PHASE_STATUS_FILE))
        completed = [p for p, s in PHASE_STATUS.items() if s == 'COMPLETE']
        print(f"[status] completed phases: {completed}")
    except Exception:
        PHASE_STATUS = {}

def save_cache():
    tmp = CACHE_FILE.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(CACHE, f, indent=1, default=str)
    tmp.replace(CACHE_FILE)

def save_phase_status():
    tmp = PHASE_STATUS_FILE.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(PHASE_STATUS, f, indent=1)
    tmp.replace(PHASE_STATUS_FILE)

def jclean(obj):
    if isinstance(obj, dict):
        return {str(k): jclean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jclean(v) for v in obj]
    if isinstance(obj, (np.floating, torch.Tensor)) and not isinstance(obj, dict):
        try:
            return float(obj)
        except Exception:
            return str(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, (float, int, str, bool)) or obj is None:
        return obj
    try:
        return float(obj)
    except Exception:
        return str(obj)

STATE = {}

# ═══════════════════════════════════════════════════════════════════════════
# [EMBEDDED SOURCE AB]
# ═══════════════════════════════════════════════════════════════════════════
_SRC_AB = r'''
import gc, os, sys, time, json, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.metrics import f1_score, recall_score
warnings.filterwarnings('ignore')

def _pip(p):
    try:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',p],
                              stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    except:
        pass
for p in ['torchaudio','soundfile']:
    try:
        __import__(p)
    except:
        _pip(p)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends,'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')


class HardEmotionDataset(Dataset):
    def __init__(self, split='train', sr=16000, max_sec=8.0, min_sec=1.5,
                 max_samples=2000, difficulty='hard', seed=None):
        self.sr = sr
        self.max_len = int(max_sec * sr)
        self.min_len = int(min_sec * sr)
        self.samples = []
        self.source = f'synthetic-{difficulty}'
        seed_map = {'train': 42, 'validation': 123, 'val': 123, 'test': 456}
        self.rng = np.random.RandomState(seed or seed_map.get(split, 42))
        if difficulty == 'hard':
            class_probs = [0.30, 0.20, 0.28, 0.22]
        else:
            class_probs = [0.25, 0.25, 0.25, 0.25]
        self.speakers = self._create_speakers(20)
        n = max_samples
        labels = self.rng.choice(4, size=n, p=class_probs)
        for i in range(n):
            emo = int(labels[i])
            speaker = self.speakers[self.rng.randint(0, len(self.speakers))]
            is_ambiguous = self.rng.random() < 0.15
            wav = self._generate_utterance(emo, speaker, is_ambiguous, difficulty)
            self.samples.append({'waveform': torch.from_numpy(wav), 'emotion': emo})
        dist = Counter(s['emotion'] for s in self.samples)
        names = {0:'neu', 1:'hap', 2:'sad', 3:'ang'}
        print(f"  [{split}] {self.source}: {len(self.samples)} samples "
              f"({', '.join(f'{names[k]}={dist.get(k,0)}' for k in range(4))})")

    def _create_speakers(self, n_speakers):
        speakers = []
        for i in range(n_speakers):
            gender = self.rng.choice(['M', 'F'])
            if gender == 'M':
                base_f0 = self.rng.uniform(85, 165)
                formant_shift = self.rng.uniform(0.85, 1.0)
            else:
                base_f0 = self.rng.uniform(165, 280)
                formant_shift = self.rng.uniform(1.0, 1.15)
            speakers.append({
                'gender': gender, 'base_f0': base_f0,
                'f0_variability': self.rng.uniform(0.08, 0.25),
                'energy_scale': self.rng.uniform(0.6, 1.4),
                'breathiness': self.rng.uniform(0.02, 0.15),
                'speaking_rate': self.rng.uniform(0.7, 1.3),
                'formant_shift': formant_shift,
                'vibrato_tendency': self.rng.uniform(0.0, 0.3),
                'jitter': self.rng.uniform(0.005, 0.03),
                'shimmer': self.rng.uniform(0.01, 0.08),
            })
        return speakers

    def _generate_utterance(self, emotion, speaker, is_ambiguous, difficulty):
        rng = self.rng
        if difficulty == 'hard':
            duration = rng.uniform(1.5, 8.0)
        else:
            duration = rng.uniform(2.0, 6.0)
        n_samples = int(duration * self.sr)
        t = np.linspace(0, duration, n_samples, dtype=np.float32)
        emo_params = {
            0: {'f0_shift': rng.uniform(-0.10, 0.10), 'energy_mod': rng.uniform(0.8, 1.2),
                'contour': 'flat', 'rate_mod': 1.0, 'noise_extra': 0.0},
            1: {'f0_shift': rng.uniform(0.0, 0.35), 'energy_mod': rng.uniform(1.0, 1.5),
                'contour': 'rising', 'rate_mod': rng.uniform(1.0, 1.3), 'noise_extra': 0.0},
            2: {'f0_shift': rng.uniform(-0.25, 0.05), 'energy_mod': rng.uniform(0.4, 0.9),
                'contour': 'falling', 'rate_mod': rng.uniform(0.6, 0.9),
                'noise_extra': rng.uniform(0.0, 0.05)},
            3: {'f0_shift': rng.uniform(0.05, 0.40), 'energy_mod': rng.uniform(1.1, 1.8),
                'contour': 'erratic', 'rate_mod': rng.uniform(1.0, 1.4),
                'noise_extra': rng.uniform(0.05, 0.15)},
        }
        params = emo_params[emotion]
        if is_ambiguous:
            other_emo = rng.choice([e for e in range(4) if e != emotion])
            other_params = emo_params[other_emo]
            blend = rng.uniform(0.3, 0.6)
            for key in ['f0_shift', 'energy_mod', 'rate_mod', 'noise_extra']:
                params[key] = (1 - blend) * params[key] + blend * other_params[key]
            if rng.random() < 0.3:
                params['contour'] = other_params['contour']
        base_f0 = speaker['base_f0'] * (1 + params['f0_shift'])
        f0_jitter = speaker['jitter'] * rng.randn(n_samples).astype(np.float32)
        kernel_size = max(3, int(0.01 * self.sr))
        if kernel_size < n_samples:
            kernel = np.ones(kernel_size, dtype=np.float32) / kernel_size
            f0_jitter = np.convolve(f0_jitter, kernel, mode='same')
        f0_contour = np.full(n_samples, base_f0, dtype=np.float32)
        if params['contour'] == 'rising':
            f0_contour *= (1 + 0.15 * t / duration)
        elif params['contour'] == 'falling':
            f0_contour *= (1 - 0.15 * t / duration)
        elif params['contour'] == 'erratic':
            n_jumps = rng.randint(3, 8)
            for _ in range(n_jumps):
                jump_pos = rng.randint(0, n_samples)
                jump_dur = rng.randint(int(0.05 * self.sr), int(0.3 * self.sr))
                jump_amount = rng.uniform(-0.2, 0.2) * base_f0
                end = min(jump_pos + jump_dur, n_samples)
                f0_contour[jump_pos:end] += jump_amount
        f0_variation = speaker['f0_variability'] * base_f0
        f0_slow_mod = f0_variation * np.sin(2 * np.pi * rng.uniform(0.5, 3.0) * t)
        f0_contour += f0_slow_mod + f0_jitter * base_f0
        f0_contour = np.clip(f0_contour, 50, 500)
        phase = np.cumsum(2 * np.pi * f0_contour / self.sr)
        energy = params['energy_mod'] * speaker['energy_scale']
        h1_amp = energy * rng.uniform(0.6, 1.0)
        h2_amp = energy * rng.uniform(0.15, 0.45)
        h3_amp = energy * rng.uniform(0.05, 0.25)
        h4_amp = energy * rng.uniform(0.02, 0.15)
        h5_amp = energy * rng.uniform(0.01, 0.08)
        wav = (h1_amp * np.sin(phase) + h2_amp * np.sin(2 * phase) +
               h3_amp * np.sin(3 * phase) + h4_amp * np.sin(4 * phase) +
               h5_amp * np.sin(5 * phase)).astype(np.float32)
        shimmer = 1 + speaker['shimmer'] * rng.randn(n_samples).astype(np.float32)
        if kernel_size < n_samples:
            shimmer = np.convolve(shimmer, kernel, mode='same')
        wav *= shimmer
        rate = params['rate_mod'] * speaker['speaking_rate']
        syllable_rate = rng.uniform(3.0, 6.0) * rate
        syllable_mod = 0.5 + 0.5 * np.sin(2 * np.pi * syllable_rate * t)
        syllable_noise = 0.2 * rng.randn(n_samples).astype(np.float32)
        if kernel_size < n_samples:
            syllable_noise = np.convolve(syllable_noise, kernel, mode='same')
        syllable_mod += syllable_noise
        syllable_mod = np.clip(syllable_mod, 0.1, 1.0)
        wav *= syllable_mod
        if emotion == 1 or (is_ambiguous and rng.random() < 0.3):
            if rng.random() < 0.6 + speaker['vibrato_tendency']:
                vib_rate = rng.uniform(4, 7)
                vib_depth = rng.uniform(0.03, 0.12)
                vib = 1 + vib_depth * np.sin(2 * np.pi * vib_rate * t)
                wav *= vib.astype(np.float32)
        if emotion == 2:
            if rng.random() < 0.5:
                decay = np.exp(-rng.uniform(0.1, 0.4) * t).astype(np.float32)
                wav *= decay
            if rng.random() < 0.4:
                n_pauses = rng.randint(1, 3)
                for _ in range(n_pauses):
                    ps = rng.randint(0, max(1, n_samples - 8000))
                    pl = rng.randint(2000, 8000)
                    fade = min(500, pl // 4)
                    end_pos = min(ps + pl, n_samples)
                    wav[ps:end_pos] *= 0.05
                    if fade > 0 and ps + fade < n_samples:
                        wav[ps:ps+fade] *= np.linspace(1, 0.05, fade).astype(np.float32)
        if emotion == 3:
            if rng.random() < 0.5:
                n_bursts = rng.randint(1, 4)
                for _ in range(n_bursts):
                    bp = rng.randint(0, max(1, n_samples - 3000))
                    bl = rng.randint(800, 3000)
                    end = min(bp + bl, n_samples)
                    wav[bp:end] *= rng.uniform(1.2, 1.6)
                    wav[bp:end] += (rng.uniform(0.05, 0.12) * energy *
                                   rng.randn(end - bp)).astype(np.float32)
        breathiness = speaker['breathiness'] + params['noise_extra']
        wav += breathiness * energy * rng.randn(n_samples).astype(np.float32)
        if difficulty == 'hard':
            channel = rng.choice(['clean', 'telephone', 'room', 'noisy'],
                                  p=[0.25, 0.25, 0.25, 0.25])
        else:
            channel = rng.choice(['clean', 'room'], p=[0.6, 0.4])
        if channel == 'telephone':
            from numpy.fft import rfft, irfft, rfftfreq
            freqs = rfftfreq(n_samples, 1.0 / self.sr)
            spectrum = rfft(wav)
            mask = np.ones_like(freqs)
            mask[freqs < 300] *= 0.1
            mask[freqs > 3400] *= 0.1
            mask[(freqs >= 200) & (freqs < 300)] *= np.linspace(
                0.1, 1.0, np.sum((freqs >= 200) & (freqs < 300)))
            mask[(freqs > 3000) & (freqs <= 3400)] *= np.linspace(
                1.0, 0.1, np.sum((freqs > 3000) & (freqs <= 3400)))
            wav = irfft(spectrum * mask, n=n_samples).astype(np.float32)
            wav += 0.02 * rng.randn(n_samples).astype(np.float32)
        elif channel == 'room':
            delays = [int(rng.uniform(0.01, 0.05) * self.sr) for _ in range(3)]
            gains = [rng.uniform(0.1, 0.3) for _ in range(3)]
            reverb = np.zeros_like(wav)
            for delay, gain in zip(delays, gains):
                if delay < n_samples:
                    reverb[delay:] += gain * wav[:n_samples - delay]
            wav += reverb
        elif channel == 'noisy':
            snr_db = rng.uniform(5, 20)
            signal_power = np.mean(wav ** 2)
            noise_power = signal_power / (10 ** (snr_db / 10))
            noise = np.sqrt(noise_power) * rng.randn(n_samples).astype(np.float32)
            if rng.random() < 0.5:
                babble_mod = 0.5 + 0.5 * np.sin(2 * np.pi * rng.uniform(1, 4) * t)
                noise *= babble_mod.astype(np.float32)
            wav += noise
        wav = wav / (np.abs(wav).max() + 1e-8)
        wav *= rng.uniform(0.3, 1.0)
        return wav.astype(np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        w = s['waveform']
        ol = w.shape[0]
        if w.shape[0] > self.max_len:
            st = torch.randint(0, w.shape[0] - self.max_len, (1,)).item()
            w = w[st:st + self.max_len]
        elif w.shape[0] < self.max_len:
            w = F.pad(w, (0, self.max_len - w.shape[0]))
        return {'waveform': w, 'emotion': s['emotion'],
                'length': min(ol, self.max_len)}


def create_dataloaders(batch_size=48, max_sec=8.0,
                       n_train=2000, n_val=500, n_test=500):
    print("\n" + "-"*55)
    print("  Creating HARD Emotion Data")
    print("-"*55)
    tds  = HardEmotionDataset('train',      max_sec=max_sec,
                              max_samples=n_train, difficulty='hard')
    vds  = HardEmotionDataset('validation', max_sec=max_sec,
                              max_samples=n_val,   difficulty='hard')
    teds = HardEmotionDataset('test',       max_sec=max_sec,
                              max_samples=n_test,  difficulty='hard')
    pin = DEVICE.type == 'cuda'
    kw = dict(batch_size=batch_size, num_workers=0,
              pin_memory=pin, drop_last=False)
    print(f"\n  Total: {len(tds)} train + {len(vds)} val + {len(teds)} test")
    print("-"*55)
    return (DataLoader(tds, shuffle=True,  **kw),
            DataLoader(vds, shuffle=False, **kw),
            DataLoader(teds, shuffle=False, **kw))


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.s = nn.Parameter(torch.ones(d)); self.e = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.e) * self.s

class MambaBlock(nn.Module):
    def __init__(self, d, drop=0.1):
        super().__init__()
        self.norm = RMSNorm(d)
        self.ip   = nn.Linear(d, d*2, bias=False)
        self.conv = nn.Conv1d(d, d, 5, padding=2, groups=d)
        self.op   = nn.Linear(d, d, bias=False)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        r = x; x = self.norm(x); x, z = self.ip(x).chunk(2, -1)
        return r + self.drop(
            self.op(self.conv(x.transpose(1,2)).transpose(1,2) * F.silu(z)))

class BiMambaBlock(nn.Module):
    def __init__(self, d, drop=0.1):
        super().__init__()
        self.f = MambaBlock(d, drop); self.b = MambaBlock(d, drop)
        self.g = nn.Linear(d*2, d)
    def forward(self, x):
        return self.g(torch.cat(
            [self.f(x), self.b(x.flip(1)).flip(1)], -1))

class HausdorffW(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.beta = nn.Parameter(torch.tensor(1.0))
    def _h(self, a, b):
        d = torch.cdist(a[:32], b[:32], p=2)
        return torch.maximum(d.min(1)[0].max(), d.min(0)[0].max())
    def forward(self, feats):
        n = len(feats)
        if n == 1:
            return torch.ones(1, device=feats[0].device), []
        hd = [self._h(feats[i], feats[i+1]) for i in range(n-1)]
        raw = [torch.exp(-self.beta * hd[0])]
        for i in range(1, n-1):
            raw.append(torch.exp(-self.beta * (hd[i-1] - hd[i]).abs()))
        raw.append(torch.exp(-self.beta * hd[-1]))
        return F.softmax(torch.stack(raw), 0), hd

class EqualW(nn.Module):
    def __init__(self, n): super().__init__()
    def forward(self, f):
        return torch.ones(len(f), device=f[0].device) / len(f), []

class LearnedW(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.l = nn.Parameter(torch.zeros(n))
    def forward(self, f):
        return F.softmax(self.l[:len(f)], 0), []

class AttPool(nn.Module):
    def __init__(self, d, o=128):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(d, d//4), nn.Tanh(), nn.Linear(d//4, 1))
        self.p = nn.Linear(d*2, o)
    def forward(self, x):
        a = F.softmax(self.a(x).squeeze(-1), 1).unsqueeze(-1)
        m = (a*x).sum(1)
        s = ((a*(x - m.unsqueeze(1)).pow(2)).sum(1)).clamp(1e-8).sqrt()
        return self.p(torch.cat([m, s], -1))

class MeanPool(nn.Module):
    def __init__(self, d, o=128):
        super().__init__()
        self.p = nn.Linear(d, o)
    def forward(self, x):
        return self.p(x.mean(1))

class Frontend(nn.Module):
    def __init__(self, ic, oc, dil=1, stride=1, use_dil=True):
        super().__init__()
        d = dil if use_dil else 1
        self.c = nn.Sequential(
            nn.Conv1d(ic, oc, 5, padding=2*d, dilation=d),
            nn.BatchNorm1d(oc), nn.GELU())
        self.p = nn.AvgPool1d(stride) if stride > 1 else nn.Identity()
    def forward(self, x):
        return self.p(self.c(x.transpose(1,2))).transpose(1,2)

class ScalePath(nn.Module):
    def __init__(self, nm, dm, nl, dil, stride,
                 bidir=True, use_dil=True, pool='attentive', drop=0.1):
        super().__init__()
        self.fe = Frontend(nm, dm, dil, stride, use_dil)
        B = BiMambaBlock if bidir else MambaBlock
        self.blks = nn.ModuleList([B(dm, drop) for _ in range(nl)])
        self.norm = RMSNorm(dm)
        self.pool = MeanPool(dm) if pool == 'mean' else AttPool(dm)
    def forward(self, x):
        x = self.fe(x)
        for b in self.blks:
            x = b(x)
        return self.pool(self.norm(x))


class FME(nn.Module):
    def __init__(self, nm=80, dm=128, du=512, ns=4, nc=4, sr=16000, drop=0.1,
                 bidir=True, wt='hausdorff', use_dil=True, pt='attentive',
                 custom_layers=None):
        super().__init__()
        self.nm = nm; self.ns = ns
        dils = (1, 4, 16, 64, 256)[:ns]
        strs = (1, 2, 4, 8, 16)[:ns]
        lyrs = custom_layers or (2, 2, 3, 3, 4)[:ns]
        self.mel_ext = self._mk_mel(sr)
        self.paths = nn.ModuleList([
            ScalePath(nm, dm, lyrs[i] if i < len(lyrs) else 2,
                      dils[i] if i < len(dils) else 1,
                      strs[i] if i < len(strs) else 1,
                      bidir, use_dil, pt, drop)
            for i in range(ns)])
        WM = {'hausdorff': HausdorffW, 'learned': LearnedW}
        self.wt = WM.get(wt, EqualW)(ns)
        self.fuse = nn.Sequential(
            nn.Linear(128*ns, du), RMSNorm(du), nn.GELU(), nn.Dropout(drop))
        self.cls = nn.Sequential(
            nn.Linear(du, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, nc))
        self._init()

    def _mk_mel(self, sr):
        try:
            import torchaudio.transforms as T
            return nn.Sequential(
                T.MelSpectrogram(sample_rate=sr, n_fft=512, hop_length=160,
                                 n_mels=self.nm, f_min=20, f_max=8000),
                T.AmplitudeToDB(stype='power', top_db=80))
        except:
            return None

    def _ext(self, wav):
        if self.mel_ext:
            wav = (wav - wav.mean(-1, keepdim=True)) / (wav.std(-1, keepdim=True) + 1e-8)
            m = self.mel_ext(wav)
            m = (m - m.mean()) / (m.std() + 1e-8)
            return m.transpose(1, 2).clamp(-10, 10)
        return torch.randn(wav.shape[0], wav.shape[1]//160, self.nm,
                           device=wav.device) * 0.1

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, wav, **kw):
        mel = self._ext(wav)
        feats = [p(mel) for p in self.paths]
        w, hd = self.wt(feats)
        z = self.fuse(torch.cat([w[i]*f for i, f in enumerate(feats)], -1))
        return {'z_unified': z,
                'predictions': {'emotion_logits': self.cls(z)},
                'weights': w.detach(), 'scale_features': feats}

    def nparams(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def ua_score(p, t):
    return recall_score(t, p, average='macro', zero_division=0) * 100

def rtf(model, loader, dev, nb=3):
    model.eval(); asec = psec = 0
    with torch.no_grad():
        for i, b in enumerate(loader):
            if i >= nb + 1: break
            w = b['waveform'].to(dev)
            l = b.get('length', torch.full((w.shape[0],), w.shape[1]))
            a = l.float().sum().item() / 16000
            if i == 0:
                _ = model(w)
                if dev.type == 'cuda': torch.cuda.synchronize()
                continue
            t0 = time.perf_counter(); _ = model(w)
            if dev.type == 'cuda': torch.cuda.synchronize()
            psec += time.perf_counter() - t0; asec += a
    return psec / max(asec, 1e-8)


@dataclass
class Cfg:
    epochs: int = 8
    lr: float = 3e-4
    wd: float = 0.01
    clip: float = 1.0
    patience: int = 4
    max_train_batch: int = 60
    max_val_batch: int = 20

class Trainer:
    def __init__(self, model, cfg, tl, vl, dev):
        self.m = model.to(dev); self.c = cfg; self.d = dev
        self.tl = tl; self.vl = vl
        self.opt = torch.optim.AdamW(
            model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        steps = cfg.epochs * min(len(tl), cfg.max_train_batch)
        self.sched = torch.optim.lr_scheduler.OneCycleLR(
            self.opt, max_lr=cfg.lr,
            total_steps=max(steps, 1), pct_start=0.15)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.best = 0; self.wait = 0

    def train_ep(self):
        self.m.train(); P = []; T = []
        for i, b in enumerate(self.tl):
            if i >= self.c.max_train_batch: break
            w = b['waveform'].to(self.d, non_blocking=True)
            e = b['emotion'].to(self.d, non_blocking=True)
            self.opt.zero_grad(set_to_none=True)
            try:
                lo = self.m(w)['predictions']['emotion_logits']
                l = self.loss_fn(lo, e)
                if not torch.isfinite(l): continue
                l.backward()
                torch.nn.utils.clip_grad_norm_(self.m.parameters(), self.c.clip)
                self.opt.step(); self.sched.step()
                P.extend(lo.argmax(-1).cpu().numpy())
                T.extend(e.cpu().numpy())
            except RuntimeError:
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        return ua_score(P, T) if T else 0

    @torch.no_grad()
    def eval(self):
        self.m.eval(); P = []; T = []
        for i, b in enumerate(self.vl):
            if i >= self.c.max_val_batch: break
            w = b['waveform'].to(self.d, non_blocking=True)
            e = b['emotion'].to(self.d, non_blocking=True)
            try:
                p = self.m(w)['predictions']['emotion_logits'].argmax(-1)
                P.extend(p.cpu().numpy()); T.extend(e.cpu().numpy())
            except: pass
        ua = ua_score(P, T) if T else 0
        f1 = f1_score(T, P, average='macro', zero_division=0) * 100 if T else 0
        return ua, f1

    def run(self, verbose=False):
        t0 = time.time()
        for ep in range(self.c.epochs):
            tua = self.train_ep()
            vua, vf1 = self.eval()
            if vua > self.best: self.best = vua; self.wait = 0
            else: self.wait += 1
            if verbose:
                print(f"      Ep{ep+1}: TrainUA={tua:.1f}% ValUA={vua:.1f}%")
            if self.wait >= self.c.patience: break
        fua, ff1 = self.eval()
        r = rtf(self.m, self.vl, self.d)
        return {'UA': fua, 'F1': ff1, 'RTF': r, 'time': time.time()-t0,
                'epochs': ep+1, 'params': self.m.nparams(), 'best_ua': self.best}
'''

NS_AB = {'__name__': 'embedded_ab'}
_t0 = time.time()
exec(compile(_SRC_AB, 'embedded_ab', 'exec'), NS_AB)
print(f"  [embedded AB] loaded in {time.time()-_t0:.1f}s")

# ═══════════════════════════════════════════════════════════════════════════
# [EMBEDDED SOURCE QP]
# ═══════════════════════════════════════════════════════════════════════════
_SRC_QP = r'''
import os, sys, json, time, math, random, copy, gc
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple, Any, Union
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    mean_squared_error, mean_absolute_error, recall_score)
from sklearn.manifold import TSNE
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm.auto import tqdm

try:
    import torchaudio
    import torchaudio.transforms as T
    TORCHAUDIO_AVAILABLE = True
except:
    TORCHAUDIO_AVAILABLE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def to_serializable(obj):
    if obj is None: return None
    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, (np.integer, np.int32, np.int64)): return int(obj)
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        v = float(obj)
        return v if not (np.isnan(v) or np.isinf(v)) else 0.0
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy().tolist()
    if isinstance(obj, (int, float, str, bool)):
        if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)):
            return 0.0
        return obj
    return str(obj)

def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(to_serializable(obj), f, indent=2)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def safe_mean(x):
    if isinstance(x, torch.Tensor):
        x = x[~torch.isnan(x)]
        return x.mean() if len(x) > 0 else torch.tensor(0.0)
    else:
        x = np.array(x); x = x[~np.isnan(x)]
        return np.mean(x) if len(x) > 0 else 0.0

class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        if not np.isnan(val) and not np.isinf(val):
            self.val = val; self.sum += val * n
            self.count += n
            self.avg = self.sum / self.count if self.count > 0 else 0


@dataclass
class Config:
    exp_name: str = "fme_stable"
    seed: int = 42
    output_dir: str = "./experiments"
    device: str = str(DEVICE)
    verbose: bool = True
    d_model: int = 96
    d_state: int = 8
    d_conv: int = 4
    expand: int = 2
    d_unified: int = 384
    num_scales: int = 4
    dropout: float = 0.1
    sample_rate: int = 16000
    n_fft: int = 512
    hop_length: int = 160
    n_mels: int = 64
    max_audio_sec: float = 6.0
    batch_size: int = 16
    num_epochs: int = 40
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    gradient_clip: float = 0.5
    use_amp: bool = False
    label_smoothing: float = 0.1
    w_emotion: float = 1.0
    w_personality: float = 0.2
    num_train: int = 3000
    num_val: int = 600
    num_test: int = 600
    num_workers: int = 0
    patience: int = 15
    min_delta: float = 0.001
    scale_configs: Tuple = (
        {'name': 'micro',  'dilation': 1,  'stride': 1, 'layers': 2},
        {'name': 'meso',   'dilation': 4,  'stride': 2, 'layers': 3},
        {'name': 'macro',  'dilation': 16, 'stride': 4, 'layers': 3},
        {'name': 'supra',  'dilation': 64, 'stride': 8, 'layers': 4},
    )
    def __post_init__(self):
        self.max_samples = int(self.max_audio_sec * self.sample_rate)
        self.exp_dir = (Path(self.output_dir) / self.exp_name /
                        datetime.now().strftime("%Y%m%d_%H%M%S"))
    def to_dict(self):
        d = asdict(self); d['scale_configs'] = list(self.scale_configs)
        return to_serializable(d)


class StableSpeechDataset(Dataset):
    EMOTIONS = ['angry', 'happy', 'sad', 'neutral']
    def __init__(self, num_samples, max_len, sample_rate=16000, seed=42):
        self.num_samples = num_samples; self.max_len = max_len
        self.sample_rate = sample_rate
        np.random.seed(seed)
        samples_per_class = num_samples // 4
        emotions = []
        for i in range(4):
            emotions.extend([i] * samples_per_class)
        remaining = num_samples - len(emotions)
        emotions.extend(np.random.randint(0, 4, remaining).tolist())
        np.random.shuffle(emotions)
        self.emotions = np.array(emotions)
        self.personalities = self._generate_personalities()
        self.lengths = np.random.randint(int(max_len * 0.5), max_len, num_samples)
    def _generate_personalities(self):
        personality = np.zeros((self.num_samples, 5), dtype=np.float32)
        correlations = {
            0: [0.4, 0.5, 0.7, 0.3, 0.8], 1: [0.7, 0.6, 0.8, 0.8, 0.3],
            2: [0.5, 0.4, 0.3, 0.5, 0.7], 3: [0.5, 0.6, 0.5, 0.5, 0.4],
        }
        for i, emo in enumerate(self.emotions):
            base = correlations[emo]
            noise = np.random.randn(5) * 0.15
            personality[i] = np.clip(base + noise, 0.1, 0.9)
        return personality
    def _generate_waveform(self, length, emotion):
        t = np.linspace(0, length / self.sample_rate, length, dtype=np.float32)
        if emotion == 0:
            f0, energy, harmonics, noise_level = (
                200+np.random.rand()*100, 0.8, [1,.8,.6,.5,.4], 0.15)
        elif emotion == 1:
            f0, energy, harmonics, noise_level = (
                180+np.random.rand()*80, 0.6, [1,.6,.4,.3,.2], 0.1)
        elif emotion == 2:
            f0, energy, harmonics, noise_level = (
                100+np.random.rand()*50, 0.3, [1,.4,.2,.1,.05], 0.05)
        else:
            f0, energy, harmonics, noise_level = (
                140+np.random.rand()*60, 0.5, [1,.5,.3,.2,.1], 0.08)
        wave = np.zeros(length, dtype=np.float32)
        for i, amp in enumerate(harmonics):
            wave += amp * np.sin(2*np.pi*f0*(i+1)*t + np.random.rand()*2*np.pi)
        attack, release = int(0.05*length), int(0.1*length)
        envelope = np.ones(length, dtype=np.float32)
        envelope[:attack] = np.linspace(0, 1, attack)
        envelope[-release:] = np.linspace(1, 0, release)
        wave *= envelope
        wave += noise_level * np.random.randn(length).astype(np.float32)
        wave *= energy
        mx = np.abs(wave).max()
        if mx > 0: wave = wave / mx * 0.9
        return wave
    def __len__(self): return self.num_samples
    def __getitem__(self, idx):
        emotion = self.emotions[idx]; length = self.lengths[idx]
        wave = self._generate_waveform(length, emotion)
        if len(wave) < self.max_len:
            wave = np.pad(wave, (0, self.max_len - len(wave)))
        return {
            'waveform': torch.from_numpy(wave).float(),
            'emotion': torch.tensor(emotion, dtype=torch.long),
            'personality': torch.from_numpy(self.personalities[idx]).float(),
            'length': torch.tensor(length, dtype=torch.long),
        }

def create_dataloaders(config):
    train_ds = StableSpeechDataset(
        config.num_train, config.max_samples, config.sample_rate, config.seed)
    val_ds   = StableSpeechDataset(
        config.num_val,   config.max_samples, config.sample_rate, config.seed+1)
    test_ds  = StableSpeechDataset(
        config.num_test,  config.max_samples, config.sample_rate, config.seed+2)
    kw = dict(num_workers=config.num_workers, pin_memory=False)
    return (DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                       drop_last=True, **kw),
            DataLoader(val_ds,   batch_size=config.batch_size, shuffle=False, **kw),
            DataLoader(test_ds,  batch_size=config.batch_size, shuffle=False, **kw))


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        x = torch.clamp(x, -1e4, 1e4)
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

class StableSSM(nn.Module):
    def __init__(self, d_model, d_state=8, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.d_inner = expand * d_model
        self.dt_rank = max(1, d_model // 16)
        self.in_proj  = nn.Linear(d_model, self.d_inner*2, bias=False)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, d_conv,
                                   padding=d_conv-1, groups=self.d_inner, bias=True)
        self.x_proj   = nn.Linear(self.d_inner, self.dt_rank + d_state*2, bias=False)
        self.dt_proj  = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        with torch.no_grad():
            self.dt_proj.bias.uniform_(0.001, 0.1)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(
            self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A + 0.001))
        self.D = nn.Parameter(torch.ones(self.d_inner) * 0.5)
        self._init_weights()
    def _init_weights(self):
        for m in [self.in_proj, self.x_proj, self.out_proj]:
            nn.init.xavier_uniform_(m.weight, gain=0.5)
        nn.init.xavier_uniform_(self.dt_proj.weight, gain=0.1)
    def forward(self, x):
        b, l, _ = x.shape
        xz = self.in_proj(x); x_ssm, z = xz.chunk(2, dim=-1)
        x_ssm = x_ssm.transpose(1, 2)
        x_ssm = self.conv1d(x_ssm)[:, :, :l].transpose(1, 2)
        x_ssm = F.silu(x_ssm)
        y = self._ssm_forward(x_ssm)
        y = y * F.silu(z)
        return self.out_proj(y)
    def _ssm_forward(self, x):
        b, l, d = x.shape; n = self.d_state
        A = -torch.exp(torch.clamp(self.A_log, -10, 2)); D = self.D
        x_dbl = self.x_proj(x)
        delta, B, C = x_dbl.split([self.dt_rank, n, n], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        delta = torch.clamp(delta, 1e-4, 1.0)
        y = torch.zeros_like(x)
        h = torch.zeros(b, d, n, device=x.device, dtype=x.dtype)
        for t in range(l):
            dt = delta[:, t, :, None]
            A_bar = torch.exp(torch.clamp(dt * A, -10, 0))
            B_bar = dt * B[:, t, None, :]
            h = A_bar * h + B_bar * x[:, t, :, None]
            h = torch.clamp(h, -100, 100)
            y[:, t] = (h * C[:, t, None, :]).sum(-1)
        y = y + x * D
        return torch.clamp(y, -100, 100)

class BiMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=8, dropout=0.1):
        super().__init__()
        self.norm    = RMSNorm(d_model)
        self.ssm_fwd = StableSSM(d_model, d_state)
        self.ssm_bwd = StableSSM(d_model, d_state)
        self.fusion  = nn.Linear(d_model*2, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.fusion.weight, gain=0.5)
    def forward(self, x):
        residual = x; x = self.norm(x)
        h_fwd = self.ssm_fwd(x)
        h_bwd = torch.flip(self.ssm_bwd(torch.flip(x, [1])), [1])
        h = self.fusion(torch.cat([h_fwd, h_bwd], dim=-1))
        return residual + self.dropout(h)

class ConvFrontend(nn.Module):
    def __init__(self, in_ch, out_ch, dilation, stride):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 5, padding=2*dilation,
                               dilation=dilation, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=dilation,
                               dilation=dilation, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.downsample = (nn.AvgPool1d(stride, stride)
                           if stride > 1 else nn.Identity())
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.gelu(self.bn1(self.conv1(x)))
        x = F.gelu(self.bn2(self.conv2(x)))
        return self.downsample(x).transpose(1, 2)

class AttentivePooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model//2), nn.Tanh(),
            nn.Linear(d_model//2, 1, bias=False))
    def forward(self, x):
        w = F.softmax(self.attn(x).squeeze(-1), dim=-1).unsqueeze(-1)
        return (w * x).sum(dim=1)

class ScalePathway(nn.Module):
    def __init__(self, n_mels, d_model, num_layers, dilation,
                 stride, d_state=8, dropout=0.1):
        super().__init__()
        self.frontend = ConvFrontend(n_mels, d_model, dilation, stride)
        self.layers = nn.ModuleList([
            BiMambaBlock(d_model, d_state, dropout) for _ in range(num_layers)])
        self.norm = RMSNorm(d_model)
        self.pool = AttentivePooling(d_model)
    def forward(self, x, return_seq=False):
        x = self.frontend(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        pooled = self.pool(x)
        return (pooled, x) if return_seq else pooled

class MelExtractor(nn.Module):
    def __init__(self, config):
        super().__init__(); self.config = config
        if TORCHAUDIO_AVAILABLE:
            self.mel = T.MelSpectrogram(
                sample_rate=config.sample_rate, n_fft=config.n_fft,
                hop_length=config.hop_length, n_mels=config.n_mels,
                f_min=20, f_max=8000)
            self.amp_to_db = T.AmplitudeToDB(stype='power', top_db=80)
        else:
            self.mel = None
    def forward(self, waveform):
        if self.mel is not None:
            waveform = waveform - waveform.mean(-1, keepdim=True)
            waveform = waveform / (waveform.std(-1, keepdim=True) + 1e-8)
            mel = self.amp_to_db(self.mel(waveform))
            mel = (mel - mel.mean()) / (mel.std() + 1e-8)
            return torch.clamp(mel, -10, 10).transpose(1, 2)
        else:
            B = waveform.shape[0]
            T_len = waveform.shape[1] // self.config.hop_length
            return torch.randn(B, T_len, self.config.n_mels,
                               device=waveform.device) * 0.1

class FractalMambaEmbedding(nn.Module):
    EMOTIONS = ['angry', 'happy', 'sad', 'neutral']
    def __init__(self, config):
        super().__init__(); self.config = config
        self.mel_extractor = MelExtractor(config)
        self.num_scales = min(config.num_scales, len(config.scale_configs))
        self.pathways = nn.ModuleList([
            ScalePathway(config.n_mels, config.d_model,
                         config.scale_configs[i]['layers'],
                         config.scale_configs[i]['dilation'],
                         config.scale_configs[i]['stride'],
                         config.d_state, config.dropout)
            for i in range(self.num_scales)])
        self.scale_logits = nn.Parameter(torch.zeros(self.num_scales))
        self.fusion = nn.Sequential(
            nn.Linear(config.d_model * self.num_scales, config.d_unified),
            RMSNorm(config.d_unified), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.d_unified, config.d_unified),
            RMSNorm(config.d_unified))
        self.emotion_head = nn.Sequential(
            nn.Linear(config.d_unified, 128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 4))
        self.personality_head = nn.Sequential(
            nn.Linear(config.d_unified, 128), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(128, 5), nn.Sigmoid())
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
    def get_scale_weights(self):
        return F.softmax(self.scale_logits, dim=0)
    def forward(self, waveform, return_all=False):
        mel = self.mel_extractor(waveform)
        scale_features, scale_seqs = [], []
        for pw in self.pathways:
            f, s = pw(mel, return_seq=True)
            scale_features.append(f); scale_seqs.append(s)
        w = self.get_scale_weights()
        concat = torch.cat([wi*fi for wi, fi in zip(w, scale_features)], dim=-1)
        z = self.fusion(concat)
        emotion_logits = self.emotion_head(z)
        personality    = self.personality_head(z)
        if torch.isnan(emotion_logits).any():
            emotion_logits = torch.zeros_like(emotion_logits)
        if torch.isnan(personality).any():
            personality = torch.ones_like(personality) * 0.5
        out = {'emotion_logits': emotion_logits, 'personality': personality,
               'scale_weights': w.detach(), 'z_unified': z}
        if return_all:
            out['scale_features'] = scale_features
            out['scale_sequences'] = scale_seqs
        return out
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__(); self.smoothing = smoothing
    def forward(self, pred, target):
        n = pred.size(-1); log_p = F.log_softmax(pred, dim=-1)
        with torch.no_grad():
            sm = torch.zeros_like(log_p).fill_(self.smoothing / (n-1))
            sm.scatter_(1, target.unsqueeze(1), 1 - self.smoothing)
        loss = -(sm * log_p).sum(-1).mean()
        return (loss if not torch.isnan(loss)
                else torch.tensor(0.0, device=pred.device, requires_grad=True))

class MultiTaskLoss(nn.Module):
    def __init__(self, w_emotion=1.0, w_personality=0.2, smoothing=0.1):
        super().__init__()
        self.w_e = w_emotion; self.w_p = w_personality
        self.emo_loss = LabelSmoothingCE(smoothing)
        self.pers_loss = nn.MSELoss()
    def forward(self, emo_logits, emo_tgt, pers_pred, pers_tgt):
        le = self.emo_loss(emo_logits, emo_tgt)
        lp = self.pers_loss(pers_pred, pers_tgt)
        if torch.isnan(lp):
            lp = torch.tensor(0.0, device=pers_pred.device)
        total = self.w_e * le + self.w_p * lp
        return total, {'total': float(total), 'emotion': float(le),
                       'personality': float(lp)}

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience = patience; self.min_delta = min_delta
        self.counter = 0; self.best = None; self.stop = False
    def __call__(self, val):
        if self.best is None: self.best = val
        elif val > self.best + self.min_delta:
            self.best = val; self.counter = 0
        else:
            self.counter += 1; self.stop = self.counter >= self.patience
        return self.stop

class Trainer:
    def __init__(self, model, config, train_loader, val_loader, test_loader):
        self.model = model.to(config.device); self.config = config
        self.train_loader = train_loader; self.val_loader = val_loader
        self.test_loader = test_loader; self.device = config.device
        self.optimizer = AdamW(model.parameters(), lr=config.learning_rate,
                               weight_decay=config.weight_decay, eps=1e-8)
        total_steps = len(train_loader) * config.num_epochs
        self.scheduler = OneCycleLR(
            self.optimizer, max_lr=config.learning_rate,
            total_steps=total_steps, pct_start=config.warmup_ratio,
            anneal_strategy='cos')
        self.criterion = MultiTaskLoss(
            config.w_emotion, config.w_personality, config.label_smoothing)
        self.scaler = GradScaler() if config.use_amp else None
        self.early_stopping = EarlyStopping(config.patience, config.min_delta)
        self.history = defaultdict(list)
        self.best_val_acc = 0; self.best_state = None
    def train_epoch(self, epoch):
        self.model.train()
        meters = {k: AverageMeter()
                  for k in ['loss', 'emo_loss', 'pers_loss', 'acc']}
        pbar = tqdm(self.train_loader,
                    desc=f"Epoch {epoch+1}/{self.config.num_epochs}")
        for batch in pbar:
            wav = batch['waveform'].to(self.device)
            emo = batch['emotion'].to(self.device)
            pers = batch['personality'].to(self.device)
            bs = wav.size(0)
            self.optimizer.zero_grad()
            out = self.model(wav)
            loss, losses = self.criterion(
                out['emotion_logits'], emo, out['personality'], pers)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), self.config.gradient_clip)
            self.optimizer.step(); self.scheduler.step()
            with torch.no_grad():
                acc = (out['emotion_logits'].argmax(-1) == emo).float().mean()
            meters['loss'].update(losses['total'], bs)
            meters['emo_loss'].update(losses['emotion'], bs)
            meters['pers_loss'].update(losses['personality'], bs)
            meters['acc'].update(float(acc), bs)
            pbar.set_postfix(loss=f"{meters['loss'].avg:.4f}",
                             acc=f"{meters['acc'].avg:.1%}")
        return {k: v.avg for k, v in meters.items()}
    @torch.no_grad()
    def evaluate(self, loader, split='val'):
        self.model.eval()
        all_preds, all_targets, all_pp, all_pt = [], [], [], []
        total_loss, n = 0, 0
        for batch in loader:
            wav = batch['waveform'].to(self.device)
            emo = batch['emotion'].to(self.device)
            pers = batch['personality'].to(self.device)
            out = self.model(wav)
            loss, _ = self.criterion(
                out['emotion_logits'], emo, out['personality'], pers)
            if not np.isnan(float(loss)):
                total_loss += float(loss)*len(wav); n += len(wav)
            preds = np.nan_to_num(
                out['emotion_logits'].argmax(-1).cpu().numpy(),
                nan=0).astype(int)
            pp = np.nan_to_num(out['personality'].cpu().numpy(), nan=0.5)
            all_preds.extend(preds); all_targets.extend(emo.cpu().numpy())
            all_pp.append(pp); all_pt.append(pers.cpu().numpy())
        preds = np.array(all_preds); targets = np.array(all_targets)
        pp = np.concatenate(all_pp); pt = np.concatenate(all_pt)
        ua = float(recall_score(targets, preds, average='macro',
                                zero_division=0) * 100)
        res = {
            f'{split}_loss': float(total_loss / max(n, 1)),
            f'{split}_acc':  float(accuracy_score(targets, preds) * 100),
            f'{split}_ua':   ua,
            f'{split}_f1_w': float(f1_score(targets, preds, average='weighted',
                                            zero_division=0) * 100),
            f'{split}_f1_m': float(f1_score(targets, preds, average='macro',
                                            zero_division=0) * 100),
            f'{split}_uar':  ua,
        }
        try:
            res[f'{split}_pers_mse'] = float(mean_squared_error(pt, pp))
            res[f'{split}_pers_mae'] = float(mean_absolute_error(pt, pp))
        except:
            res[f'{split}_pers_mse'] = 0.0; res[f'{split}_pers_mae'] = 0.0
        for i, t in enumerate('OCEAN'):
            try:
                r, _ = pearsonr(pt[:, i], pp[:, i])
                res[f'{split}_pers_{t}_r'] = float(r) if not np.isnan(r) else 0.0
            except:
                res[f'{split}_pers_{t}_r'] = 0.0
        return res
    def train(self):
        print(f"\n{'='*60}\nTRAINING - {self.model.get_num_params():,} parameters\n{'='*60}")
        for epoch in range(self.config.num_epochs):
            tm = self.train_epoch(epoch)
            vm = self.evaluate(self.val_loader, 'val')
            for k, v in {**tm, **vm}.items():
                self.history[k].append(float(v) if not np.isnan(v) else 0.0)
            print(f"Epoch {epoch+1}: Loss={tm['loss']:.4f}  "
                  f"Acc={tm['acc']:.1%}  Val Acc={vm['val_acc']:.2f}%  "
                  f"Val UA={vm['val_ua']:.2f}%  Val F1={vm['val_f1_w']:.2f}%")
            if vm['val_acc'] > self.best_val_acc:
                self.best_val_acc = vm['val_acc']
                self.best_state = copy.deepcopy(self.model.state_dict())
                print(f"  → New best: {self.best_val_acc:.2f}%")
            if self.early_stopping(vm['val_acc']):
                print(f"\nEarly stopping at epoch {epoch+1}"); break
        if self.best_state:
            self.model.load_state_dict(self.best_state)
        test_m = self.evaluate(self.test_loader, 'test')
        print(f"\n{'='*60}\nFINAL TEST RESULTS\n{'='*60}")
        print(f"  Accuracy (WA):   {test_m['test_acc']:.2f}%")
        print(f"  UA (Recall Mac): {test_m['test_ua']:.2f}%")
        print(f"  F1 Weighted:     {test_m['test_f1_w']:.2f}%")
        print(f"  F1 Macro:        {test_m['test_f1_m']:.2f}%")
        print(f"{'='*60}")
        return dict(self.history), test_m


class Evaluator:
    def __init__(self, model, config):
        self.model = model.to(config.device); self.config = config
        self.device = config.device
    @torch.no_grad()
    def get_predictions(self, loader):
        self.model.eval(); data = defaultdict(list)
        for batch in tqdm(loader, desc="Predictions"):
            wav = batch['waveform'].to(self.device)
            out = self.model(wav, return_all=True)
            probs = F.softmax(out['emotion_logits'], dim=-1)
            preds = np.nan_to_num(
                probs.argmax(-1).cpu().numpy(), nan=0).astype(int)
            pp = np.nan_to_num(out['personality'].cpu().numpy(), nan=0.5)
            data['preds'].extend(preds)
            data['probs'].append(probs.cpu().numpy())
            data['targets'].extend(batch['emotion'].numpy())
            data['pers_pred'].append(pp)
            data['pers_tgt'].append(batch['personality'].numpy())
            data['embeddings'].append(
                np.nan_to_num(out['z_unified'].cpu().numpy(), nan=0))
        return {k: (np.array(v) if k in ('preds','targets')
                    else np.concatenate(v)) for k, v in data.items()}
    def compute_metrics(self, preds):
        t, p = preds['targets'], preds['preds']
        m = {
            'accuracy':    float(accuracy_score(t, p) * 100),
            'ua':          float(recall_score(t, p, average='macro',
                                              zero_division=0) * 100),
            'f1_weighted': float(f1_score(t, p, average='weighted',
                                          zero_division=0) * 100),
            'f1_macro':    float(f1_score(t, p, average='macro',
                                          zero_division=0) * 100),
        }
        try:
            m['pers_mse'] = float(mean_squared_error(
                preds['pers_tgt'], preds['pers_pred']))
            m['pers_mae'] = float(mean_absolute_error(
                preds['pers_tgt'], preds['pers_pred']))
        except:
            m['pers_mse'] = m['pers_mae'] = 0.0
        for i, emo in enumerate(['angry','happy','sad','neutral']):
            mask = t == i
            if mask.sum() > 0:
                m[f'{emo}_acc'] = float((p[mask] == i).mean() * 100)
        for i, tr in enumerate('OCEAN'):
            try:
                r, _ = pearsonr(preds['pers_tgt'][:, i],
                                preds['pers_pred'][:, i])
                m[f'pers_{tr}_r'] = float(r) if not np.isnan(r) else 0.0
            except:
                m[f'pers_{tr}_r'] = 0.0
        return m
    @torch.no_grad()
    def benchmark_efficiency(self, durations=[1, 5, 10, 30, 60]):
        self.model.eval(); results = {}
        for dur in durations:
            samples = int(dur * self.config.sample_rate)
            wave = torch.randn(1, samples, device=self.device)
            for _ in range(3):
                _ = self.model(wave)
                if str(self.device).startswith('cuda'):
                    torch.cuda.synchronize()
            times = []
            for _ in range(10):
                t0 = time.perf_counter()
                _ = self.model(wave)
                if str(self.device).startswith('cuda'):
                    torch.cuda.synchronize()
                times.append(time.perf_counter() - t0)
            mem = 0.0
            if str(self.device).startswith('cuda'):
                torch.cuda.reset_peak_memory_stats()
                _ = self.model(wave)
                torch.cuda.synchronize()
                mem = torch.cuda.max_memory_allocated() / 1024**2
            avg_time = float(np.mean(times))
            rtf = avg_time / dur
            results[f'{dur}s'] = {
                'time_ms':   round(avg_time * 1000, 2),
                'std_ms':    round(float(np.std(times)) * 1000, 2),
                'rtf':       round(rtf, 6),
                'memory_mb': round(mem, 1),
            }
        return results
    @torch.no_grad()
    def analyze_scales(self, loader):
        self.model.eval()
        scale_acts = [[] for _ in range(self.model.num_scales)]
        emotions = []
        for batch in loader:
            wav = batch['waveform'].to(self.device)
            out = self.model(wav, return_all=True)
            for i, f in enumerate(out['scale_features']):
                scale_acts[i].append(f.cpu().numpy())
            emotions.extend(batch['emotion'].numpy())
        scale_acts = [np.nan_to_num(np.concatenate(a), nan=0)
                      for a in scale_acts]
        emotions = np.array(emotions)
        w = self.model.get_scale_weights().cpu().numpy()
        res = {'weights': w.tolist()}
        for i in range(self.model.num_scales):
            res[f'scale_{i}_norm'] = float(
                np.linalg.norm(scale_acts[i], axis=1).mean())
            for e in range(4):
                mask = emotions == e
                if mask.sum() > 0:
                    res[f'scale_{i}_emo_{e}'] = float(
                        np.linalg.norm(scale_acts[i][mask], axis=1).mean())
        return res


class BaselineComparison:
    BASELINES = {
        'wav2vec 2.0 Base':  {'params': '95M',   'acc': 63.43, 'ua': 61.20,
                              'f1': 62.80, 'rtf': 0.045, 'complexity': 'O(n²)'},
        'HuBERT Base':       {'params': '95M',   'acc': 64.92, 'ua': 63.10,
                              'f1': 64.30, 'rtf': 0.044, 'complexity': 'O(n²)'},
        'WavLM Base':        {'params': '95M',   'acc': 65.94, 'ua': 64.50,
                              'f1': 65.20, 'rtf': 0.046, 'complexity': 'O(n²)'},
        'WavLM Large':       {'params': '317M',  'acc': 68.23, 'ua': 66.80,
                              'f1': 67.50, 'rtf': 0.120, 'complexity': 'O(n²)'},
        'emotion2vec':       {'params': '~300M', 'acc': 72.10, 'ua': 70.50,
                              'f1': 71.30, 'rtf': 0.110, 'complexity': 'O(n²)'},
    }
    def __init__(self, fme_results: dict, fme_efficiency: dict = None):
        self.fme = fme_results; self.eff = fme_efficiency or {}
    def _get_fme_rtf(self) -> float:
        if '10s' in self.eff: return self.eff['10s']['rtf']
        if self.eff:
            return float(np.mean([v['rtf'] for v in self.eff.values()]))
        return 0.0
    def print_comparison(self):
        fme_acc = self.fme.get('accuracy', 0)
        fme_ua  = self.fme.get('ua', self.fme.get('uar', 0))
        fme_f1  = self.fme.get('f1_weighted', self.fme.get('f1', 0))
        fme_rtf = self._get_fme_rtf()
        fme_par = self.fme.get('params', 0)
        p_str = (f"{fme_par/1e6:.1f}M"
                 if isinstance(fme_par, (int,float)) and fme_par > 0
                 else str(fme_par))
        hdr = (f"{'Model':<25} {'Params':<10} {'Compl.':<8} "
               f"{'Acc(%)':<9} {'UA(%)':<9} {'F1(%)':<9} {'RTF':>8}")
        sep = "-" * len(hdr)
        print("\n" + "=" * len(hdr))
        print("  BENCHMARK COMPARISON  —  UA | F1 | RTF | Accuracy")
        print("=" * len(hdr))
        print(hdr); print(sep)
        for name, info in self.BASELINES.items():
            print(f"{name:<25} {info['params']:<10} {info['complexity']:<8} "
                  f"{info['acc']:<9.2f} {info['ua']:<9.2f} {info['f1']:<9.2f} "
                  f"{info['rtf']:>8.4f}")
        print(sep)
        print(f"{'FME (Ours)':<25} {p_str:<10} {'O(n)':<8} "
              f"{fme_acc:<9.2f} {fme_ua:<9.2f} {fme_f1:<9.2f} {fme_rtf:>8.4f}")
        print("=" * len(hdr))
        best_bl_acc = max(v['acc'] for v in self.BASELINES.values())
        best_bl_rtf = min(v['rtf'] for v in self.BASELINES.values())
        print(f"\n  Δ Accuracy vs best baseline : {fme_acc - best_bl_acc:+.2f}%")
        print(f"  RTF speed-up vs fastest base: "
              f"{best_bl_rtf / max(fme_rtf, 1e-9):.1f}×  (lower RTF = faster)")
        print()
    def plot_comparison(self, save_path=None):
        names   = list(self.BASELINES.keys()) + ['FME (Ours)']
        fme_acc = self.fme.get('accuracy', 0)
        fme_ua  = self.fme.get('ua', self.fme.get('uar', 0))
        fme_f1  = self.fme.get('f1_weighted', self.fme.get('f1', 0))
        fme_rtf = self._get_fme_rtf()
        accs = [v['acc'] for v in self.BASELINES.values()] + [fme_acc]
        uas  = [v['ua']  for v in self.BASELINES.values()] + [fme_ua]
        f1s  = [v['f1']  for v in self.BASELINES.values()] + [fme_f1]
        rtfs = [v['rtf'] for v in self.BASELINES.values()] + [fme_rtf]
        colors = ['#95a5a6'] * len(self.BASELINES) + ['#2ECC71']
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('FME Benchmark — UA · F1 · RTF · Accuracy',
                     fontsize=15, fontweight='bold')
        def bar(ax, vals, ylabel, title, fmt='.2f'):
            bars = ax.barh(names, vals, color=colors)
            ax.set_xlabel(ylabel); ax.set_title(title); ax.invert_yaxis()
            for b, v in zip(bars, vals):
                ax.text(b.get_width() + 0.3,
                        b.get_y() + b.get_height()/2,
                        f'{v:{fmt}}', va='center', fontsize=9)
            ax.grid(axis='x', alpha=0.3)
        bar(axes[0,0], accs, 'Accuracy (%)', 'Weighted Accuracy')
        bar(axes[0,1], uas,  'UA (%)',       'Unweighted Accuracy (UA)')
        bar(axes[1,0], f1s,  'F1 (%)',       'F1 Score (Weighted)')
        ax = axes[1,1]
        rtf_colors = (['#E74C3C' if r > fme_rtf else '#95a5a6'
                       for r in rtfs[:-1]] + ['#2ECC71'])
        bars2 = ax.barh(names, rtfs, color=rtf_colors)
        ax.set_xlabel('RTF (lower = faster)')
        ax.set_title('Real-Time Factor (RTF)'); ax.invert_yaxis()
        for b, v in zip(bars2, rtfs):
            ax.text(b.get_width() + 0.001,
                    b.get_y() + b.get_height()/2,
                    f'{v:.4f}', va='center', fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        if save_path:
            fig.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
    def to_dataframe(self) -> pd.DataFrame:
        rows = []
        for name, info in self.BASELINES.items():
            rows.append({'Model': name, 'Params': info['params'],
                         'Complexity': info['complexity'],
                         'Acc(%)': info['acc'], 'UA(%)': info['ua'],
                         'F1(%)': info['f1'], 'RTF': info['rtf']})
        fme_par = self.fme.get('params', 0)
        rows.append({
            'Model': 'FME (Ours)',
            'Params': (f"{fme_par/1e6:.1f}M"
                       if isinstance(fme_par,(int,float)) else str(fme_par)),
            'Complexity': 'O(n)',
            'Acc(%)': self.fme.get('accuracy', 0),
            'UA(%)':  self.fme.get('ua', self.fme.get('uar', 0)),
            'F1(%)':  self.fme.get('f1_weighted', self.fme.get('f1', 0)),
            'RTF':    self._get_fme_rtf(),
        })
        return pd.DataFrame(rows)


class Visualizer:
    EMOTIONS = ['Angry', 'Happy', 'Sad', 'Neutral']
    COLORS   = ['#E74C3C', '#F39C12', '#3498DB', '#95A5A6']
    SCALES   = ['Micro', 'Meso', 'Macro', 'Supra']
    def __init__(self, save_dir=None):
        self.save_dir = Path(save_dir) if save_dir else None
        if self.save_dir: self.save_dir.mkdir(parents=True, exist_ok=True)
    def _save(self, fig, name):
        if self.save_dir:
            fig.savefig(self.save_dir / name, dpi=300, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
    def plot_training(self, history, name='training.pdf'):
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(history.get('loss', []), label='Train')
        axes[0].plot(history.get('val_loss', []), label='Val')
        axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss')
        axes[0].legend(); axes[0].grid(alpha=.3)
        tr_acc = [a*100 if a < 1 else a for a in history.get('acc', [])]
        axes[1].plot(tr_acc, label='Train')
        axes[1].plot(history.get('val_acc', []), label='Val')
        axes[1].set(xlabel='Epoch', ylabel='Accuracy(%)', title='Accuracy')
        axes[1].legend(); axes[1].grid(alpha=.3)
        axes[2].plot(history.get('val_ua',
                     history.get('val_uar', [])), label='UA')
        axes[2].plot(history.get('val_f1_w', []), label='F1-W')
        axes[2].set(xlabel='Epoch', ylabel='%', title='Val UA / F1')
        axes[2].legend(); axes[2].grid(alpha=.3)
        plt.tight_layout(); self._save(fig, name)
    def plot_confusion(self, preds, name='confusion.pdf'):
        from mpl_toolkits.axes_grid1 import make_axes_locatable
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(preds['targets'], preds['preds'])
        cm_n = cm / (cm.sum(1, keepdims=True) + 1e-8)
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(cm_n, cmap='Blues')
        cax = make_axes_locatable(ax).append_axes("right", size="5%", pad=0.1)
        plt.colorbar(im, cax=cax)
        ax.set(xticks=range(4), yticks=range(4),
               xticklabels=self.EMOTIONS, yticklabels=self.EMOTIONS,
               xlabel='Predicted', ylabel='True', title='Confusion Matrix')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        for i in range(4):
            for j in range(4):
                c = 'white' if cm_n[i,j]>.5 else 'black'
                ax.text(j, i, f'{cm[i,j]}\n({cm_n[i,j]:.0%})',
                        ha='center', va='center', color=c, fontsize=10)
        plt.tight_layout(); self._save(fig, name)
    def plot_efficiency(self, eff, name='efficiency.pdf'):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        durs = [float(k.replace('s','')) for k in eff]
        fme_t = [v['time_ms'] for v in eff.values()]
        rtfs  = [v['rtf'] for v in eff.values()]
        trans_t = [50*d**2 for d in durs]
        ax = axes[0]
        ax.plot(durs, fme_t, 'o-', lw=2.5, ms=8, color='#2ECC71',
                label='FME (Ours)')
        ax.plot(durs, trans_t, 's--', lw=2.5, ms=8, color='#E74C3C',
                label='Transformer (est.)')
        ax.set(xlabel='Duration (s)', ylabel='Time (ms)', title='Inference Time')
        ax.set_yscale('log'); ax.legend(); ax.grid(alpha=.3)
        ax = axes[1]
        ax.bar(durs, rtfs, color='#3498DB', alpha=.85,
               width=[d*.15 for d in durs])
        ax.axhline(1.0, ls='--', color='red', lw=1, label='Real-time (RTF=1)')
        ax.set(xlabel='Duration (s)', ylabel='RTF',
               title='Real-Time Factor (lower = faster)')
        for d, r in zip(durs, rtfs):
            ax.text(d, r + 0.0005, f'{r:.4f}', ha='center', fontsize=9)
        ax.legend(); ax.grid(alpha=.3, axis='y')
        plt.tight_layout(); self._save(fig, name)
    def plot_scales(self, scale_res, name='scales.pdf'):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        weights = scale_res.get('weights', [.25]*4); n = len(weights)
        colors = plt.cm.viridis(np.linspace(.2, .8, n))
        bars = axes[0].bar(self.SCALES[:n], weights, color=colors)
        axes[0].set(ylabel='Weight', title='Learned Scale Weights')
        axes[0].set_ylim(0, max(weights)*1.3+.1)
        for b, w in zip(bars, weights):
            axes[0].text(b.get_x()+b.get_width()/2,
                         b.get_height()+.01, f'{w:.3f}', ha='center')
        x = np.arange(4); wd = 0.2
        for i in range(n):
            acts = [scale_res.get(f'scale_{i}_emo_{e}', 0) for e in range(4)]
            axes[1].bar(x+(i-(n-1)/2)*wd, acts, wd,
                        label=self.SCALES[i], color=colors[i], alpha=.85)
        axes[1].set(xlabel='Emotion', ylabel='Activation',
                    title='Scale Activations by Emotion', xticks=x)
        axes[1].set_xticklabels(self.EMOTIONS)
        axes[1].legend(ncol=2); axes[1].grid(alpha=.3, axis='y')
        plt.tight_layout(); self._save(fig, name)


def run_ablation(config, train_loader, val_loader, test_loader,
                 scale_values=[1,2,3,4], epochs=20):
    print(f"\n{'='*60}\nABLATION: Number of Scales\n{'='*60}")
    results = {}
    for ns in scale_values:
        print(f"\n>>> {ns} scale(s)...")
        cfg = copy.deepcopy(config)
        cfg.num_scales = ns; cfg.num_epochs = epochs; cfg.verbose = False
        model = FractalMambaEmbedding(cfg)
        trainer = Trainer(model, cfg, train_loader, val_loader, test_loader)
        _, tm = trainer.train()
        results[ns] = {
            'accuracy': float(tm['test_acc']),
            'ua':       float(tm['test_ua']),
            'f1':       float(tm['test_f1_w']),
            'params':   int(model.get_num_params()),
        }
        print(f"   Acc={results[ns]['accuracy']:.2f}%  "
              f"UA={results[ns]['ua']:.2f}%  F1={results[ns]['f1']:.2f}%")
        del model, trainer; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results


def run_experiment(
    exp_name="fme_stable", seed=42, num_epochs=40, batch_size=16,
    learning_rate=1e-4, num_scales=4, d_model=96, d_unified=384,
    num_train=3000, num_val=600, num_test=600,
    run_ablation_study=True, ablation_epochs=5, verbose=True,
):
    print("\n" + "="*70)
    print("   FRACTAL MAMBA EMBEDDING — STABLE EXPERIMENT")
    print("="*70)
    config = Config(
        exp_name=exp_name, seed=seed, num_epochs=num_epochs,
        batch_size=batch_size, learning_rate=learning_rate,
        num_scales=num_scales, d_model=d_model, d_unified=d_unified,
        num_train=num_train, num_val=num_val, num_test=num_test,
        verbose=verbose,
    )
    set_seed(config.seed)
    config.exp_dir.mkdir(parents=True, exist_ok=True)
    save_json(config.to_dict(), config.exp_dir / 'config.json')
    results = {}
    print("\n[1/7] Creating datasets...")
    train_loader, val_loader, test_loader = create_dataloaders(config)
    print(f"  Train: {len(train_loader.dataset)} | "
          f"Val: {len(val_loader.dataset)} | "
          f"Test: {len(test_loader.dataset)}")
    print("\n[2/7] Creating model...")
    model = FractalMambaEmbedding(config)
    n_params = model.get_num_params()
    print(f"  Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
    results['params'] = n_params
    print("\n[3/7] Training...")
    trainer = Trainer(model, config, train_loader, val_loader, test_loader)
    history, test_metrics = trainer.train()
    results['history']      = to_serializable(history)
    results['test_metrics'] = to_serializable(test_metrics)
    print("\n[4/7] Evaluation...")
    evaluator = Evaluator(model, config)
    preds   = evaluator.get_predictions(test_loader)
    metrics = evaluator.compute_metrics(preds)
    results['metrics'] = to_serializable(metrics)
    print(f"\n  Accuracy : {metrics['accuracy']:.2f}%")
    print(f"  UA       : {metrics['ua']:.2f}%")
    print(f"  F1 (W)   : {metrics['f1_weighted']:.2f}%")
    print(f"  F1 (M)   : {metrics['f1_macro']:.2f}%")
    print("\n[5/7] Efficiency benchmark...")
    eff = evaluator.benchmark_efficiency([1, 5, 10, 30, 60])
    results['efficiency'] = to_serializable(eff)
    print("  Duration |  Time (ms)  |     RTF")
    print("  " + "-"*38)
    for k, v in eff.items():
        print(f"    {k:>4s}   |  {v['time_ms']:>8.2f}   |  {v['rtf']:.6f}")
    scale_res = evaluator.analyze_scales(test_loader)
    results['scale_analysis'] = to_serializable(scale_res)
    ablation_res = {}
    if run_ablation_study:
        print("\n[6/7] Ablation study...")
        ablation_res = run_ablation(
            config, train_loader, val_loader, test_loader,
            scale_values=[1, 2, 3], epochs=ablation_epochs)
        results['ablation'] = to_serializable(ablation_res)
    else:
        print("\n[6/7] Skipping ablation...")
    print("\n[7/7] Visualizations & Benchmark...")
    fig_dir = str(config.exp_dir / 'figures')
    viz = Visualizer(save_dir=fig_dir)
    viz.plot_training(history, 'training.pdf')
    viz.plot_confusion(preds, 'confusion.pdf')
    viz.plot_efficiency(eff, 'efficiency.pdf')
    viz.plot_scales(scale_res, 'scales.pdf')
    benchmark = BaselineComparison(
        fme_results    = {**metrics, 'params': n_params},
        fme_efficiency = eff,
    )
    benchmark.print_comparison()
    benchmark.plot_comparison(save_path=f"{fig_dir}/benchmark.pdf")
    df = benchmark.to_dataframe()
    print("\n" + df.to_string(index=False))
    results['benchmark_table'] = to_serializable(df.to_dict('records'))
    print(f"\n{'='*60}\nSAVING RESULTS\n{'='*60}")
    torch.save({'model_state_dict': model.state_dict(),
                'config': config.to_dict(),
                'metrics': to_serializable(metrics)},
               config.exp_dir / 'model.pt')
    save_json(results, config.exp_dir / 'results.json')
    print(f"  Saved to: {config.exp_dir}")
    fme_rtf = eff.get('10s', list(eff.values())[0])['rtf']
    print(f"\n{'='*70}")
    print("   EXPERIMENT COMPLETE")
    print(f"{'='*70}")
    print(f"  Accuracy : {metrics['accuracy']:.2f}%")
    print(f"  UA       : {metrics['ua']:.2f}%")
    print(f"  F1       : {metrics['f1_weighted']:.2f}%")
    print(f"  RTF (10s): {fme_rtf:.6f}")
    print(f"  Params   : {n_params/1e6:.2f}M")
    print(f"{'='*70}")
    results['model'] = model
    return results
'''

NS_QP = {'__name__': 'embedded_qp'}
_t0 = time.time()
exec(compile(_SRC_QP, 'embedded_qp', 'exec'), NS_QP)
print(f"  [embedded QP] loaded in {time.time()-_t0:.1f}s")

# ═══════════════════════════════════════════════════════════════════════════
# [EMBEDDED SOURCE H — Hurst Regularizer]
# ═══════════════════════════════════════════════════════════════════════════
_SRC_H = r'''
class HurstExponentEstimator(nn.Module):
    def __init__(self, temperature: float = 10.0):
        super().__init__(); self.temperature = temperature
    def _soft_max(self, x: torch.Tensor, dim: int = -1) -> torch.Tensor:
        return torch.logsumexp(self.temperature * x, dim=dim) / self.temperature
    def _soft_min(self, x: torch.Tensor, dim: int = -1) -> torch.Tensor:
        return -self._soft_max(-x, dim=dim)
    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        trajectory = sequence.mean(dim=-1)
        batch, seq_len = trajectory.shape
        if seq_len < 16:
            return torch.ones(batch, device=sequence.device) * 0.7
        segment_lengths = sorted(set([
            max(4, seq_len // d) for d in [16, 8, 4, 2]]))
        if len(segment_lengths) < 2:
            return torch.ones(batch, device=sequence.device) * 0.7
        log_rs, log_n = [], []
        for n in segment_lengths:
            num_seg = seq_len // n
            if num_seg < 1: continue
            segments = trajectory[:, :num_seg * n].view(batch, num_seg, n)
            Y = segments - segments.mean(dim=-1, keepdim=True)
            Z = Y.cumsum(dim=-1)
            R = self._soft_max(Z, dim=-1) - self._soft_min(Z, dim=-1)
            S = segments.std(dim=-1).clamp(min=1e-8)
            log_rs.append(torch.log((R / S).mean(dim=-1).clamp(min=1e-8)))
            log_n.append(math.log(n))
        if len(log_rs) < 2:
            return torch.ones(batch, device=sequence.device) * 0.7
        log_rs = torch.stack(log_rs, dim=1)
        log_n  = torch.tensor(log_n, device=sequence.device)
        log_n_mean  = log_n.mean()
        log_rs_mean = log_rs.mean(dim=1, keepdim=True)
        covariance = ((log_rs - log_rs_mean) * (log_n - log_n_mean)).sum(dim=1)
        variance   = ((log_n - log_n_mean) ** 2).sum()
        H = (covariance / (variance + 1e-8)).clamp(0.1, 0.9)
        return H


class FractalRegularization(nn.Module):
    def __init__(self, target_hurst: float = 0.7, num_scales: int = 5,
                 temperature: float = 10.0):
        super().__init__()
        self.target_hurst    = target_hurst
        self.num_scales      = num_scales
        self.hurst_estimator = HurstExponentEstimator(temperature)
    def forward(self, scale_sequences, scale_features, stride_rates):
        device = scale_sequences[0].device
        hurst_estimates = []
        for seq in scale_sequences:
            if seq.shape[1] >= 8:
                H = self.hurst_estimator(seq).mean()
            else:
                H = torch.tensor(self.target_hurst, device=device)
            hurst_estimates.append(H)
        mean_hurst = torch.stack(hurst_estimates).mean()
        hurst_loss = torch.abs(mean_hurst - self.target_hurst)
        scaling_loss = torch.tensor(0.0, device=device)
        for s in range(self.num_scales - 1):
            norm_s  = scale_features[s].norm(dim=1).mean()
            norm_s1 = scale_features[s + 1].norm(dim=1).mean().clamp(min=1e-8)
            expected = (stride_rates[s] / stride_rates[s + 1]) ** self.target_hurst
            scaling_loss = scaling_loss + torch.abs(
                torch.log(norm_s / norm_s1) - math.log(expected))
        scaling_loss = scaling_loss / (self.num_scales - 1)
        total_loss = hurst_loss + 0.5 * scaling_loss
        metrics = {
            'hurst_mean':   mean_hurst.detach(),
            'hurst_loss':   hurst_loss.detach(),
            'scaling_loss': scaling_loss.detach(),
        }
        return total_loss, metrics
'''

NS_H = {'__name__': 'embedded_h'}
import math as _math_h
from typing import List, Tuple, Dict, Optional
NS_H.update({'math': _math_h, 'List': List, 'Tuple': Tuple, 'Dict': Dict,
             'Optional': Optional, 'np': np, 'torch': torch, 'nn': nn, 'F': F})
_t0 = time.time()
exec(compile(_SRC_H, 'embedded_h', 'exec'), NS_H)
print(f"  [embedded H] loaded in {time.time()-_t0:.1f}s")

print("\n" + "=" * 74)
print("  CELL 1 COMPLETE — Infrastructure loaded.")
print(f"  Cache: {len(CACHE)} jobs from previous sessions.")
print(f"  Phase status: {dict(PHASE_STATUS)}")
print(f"  CURRENT_PHASE = '{CURRENT_PHASE}'")
print("  Now run Cell 2 to execute this phase.")
print("=" * 74)

# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — EXECUTE CURRENT_PHASE AND PRINT FULL RESULTS
# Run this cell after Cell 1. Change CURRENT_PHASE in Cell 1 between sessions.
# ═══════════════════════════════════════════════════════════════════════════

# ── Verify Cell 1 has been run ───────────────────────────────────────────────
for _required in ['CACHE', 'CACHE_DIR', 'CACHE_FILE', 'PHASE_STATUS',
                  'PHASE_STATUS_FILE', 'NS_AB', 'NS_QP', 'NS_H',
                  'CURRENT_PHASE', 'DEVICE', 'SMOKE']:
    if _required not in dir():
        raise RuntimeError(
            f"Variable '{_required}' not found. Please run Cell 1 first.")

print("=" * 74)
print(f"  EXECUTING PHASE: {CURRENT_PHASE}")
print(f"  device={DEVICE}  SMOKE={SMOKE}")
print(f"  cache has {len(CACHE)} jobs from previous sessions")
print("=" * 74)

# ── Pull frequently-used names from embedded namespaces ─────────────────────
FME_AB    = NS_AB['FME']
TrainerAB = NS_AB['Trainer']
CfgAB     = NS_AB['Cfg']
rtf_ab    = NS_AB['rtf']
FractalReg = NS_H['FractalRegularization']

# ═══════════════════════════════════════════════════════════════════════════
# PHASE A — published ablation suite
# ═══════════════════════════════════════════════════════════════════════════
JOB_KW_A = {
    'scales':        {'1': {'ns': 1}, '2': {'ns': 2},
                      '3': {'ns': 3}, '4': {'ns': 4}},
    'd_model':       {'64': {'dm': 64}, '96': {'dm': 96}, '128': {'dm': 128}},
    'bidirectional': {'BiMamba': {'bidir': True}, 'UniMamba': {'bidir': False}},
    'weighting':     {'hausdorff': {'wt': 'hausdorff'},
                      'learned':   {'wt': 'learned'},
                      'equal':     {'wt': 'equal'}},
    'components':    {'full': {}, 'no_dilated_conv': {'use_dil': False},
                      'unidirectional': {'bidir': False},
                      'mean_pool': {'pt': 'mean'}},
}
ORDER_A = (
    [('scales',        k) for k in ['1', '2', '3', '4']] +
    [('d_model',       k) for k in ['64', '96', '128']] +
    [('bidirectional', k) for k in ['BiMamba', 'UniMamba']] +
    [('weighting',     k) for k in ['hausdorff', 'learned', 'equal']] +
    [('components',    k) for k in ['full', 'no_dilated_conv',
                                     'unidirectional', 'mean_pool']]
)
PUBLISHED_A = {
    'scales':        {'1': (60.0, 59.1), '2': (59.0, 57.9),
                      '3': (57.7, 55.9), '4': (58.0, 55.8)},
    'd_model':       {'64': (54.8, 52.2), '96': (56.8, 55.2),
                      '128': (59.5, 58.4)},
    'bidirectional': {'BiMamba': (58.6, 57.2), 'UniMamba': (56.9, 55.2)},
    'weighting':     {'hausdorff': (58.4, 57.6), 'learned': (57.9, 56.5),
                      'equal': (58.8, 57.0)},
    'components':    {'full': (59.3, 57.3), 'no_dilated_conv': (60.3, 58.5),
                      'unidirectional': (58.2, 56.8),
                      'mean_pool': (55.6, 54.1)},
}
PAIRS_A = [
    ('scales', '1', 'scales', '4',
     'R2-2  single-scale vs 4-scale'),
    ('scales', '4', 'scales', '2',
     'context 4-scale vs 2-scale'),
    ('weighting', 'hausdorff', 'weighting', 'equal',
     'R2-5  Hausdorff vs equal weighting'),
    ('weighting', 'hausdorff', 'weighting', 'learned',
     'R2-5  Hausdorff vs learned weighting'),
    ('bidirectional', 'BiMamba', 'bidirectional', 'UniMamba',
     'R2-15 BiMamba vs UniMamba'),
    ('components', 'full', 'components', 'no_dilated_conv',
     'R2-6  full vs no-dilated-conv'),
    ('components', 'full', 'components', 'mean_pool',
     'Table7 full vs mean-pool'),
    ('components', 'full', 'components', 'unidirectional',
     'R2-15 full vs unidirectional (Table 7)'),
    ('d_model', '128', 'd_model', '64',
     'Table4 d=128 vs d=64'),
    ('scales', '4', 'd_model', '128',
     'R2-13 same-config across blocks: S=4 vs d=128'),
    ('scales', '4', 'bidirectional', 'BiMamba',
     'R2-13 same-config across blocks: S=4 vs Bi'),
    ('scales', '4', 'weighting', 'hausdorff',
     'R2-13 same-config across blocks: S=4 vs Hausdorff'),
    ('scales', '4', 'components', 'full',
     'R2-13 same-config across blocks: S=4 vs full'),
]

def get_data_A():
    if 'A' not in STATE:
        t0 = time.time()
        if SMOKE:
            loaders = NS_AB['create_dataloaders'](
                batch_size=8, max_sec=2.0,
                n_train=16, n_val=16, n_test=16)
        else:
            loaders = NS_AB['create_dataloaders'](
                batch_size=48, max_sec=8.0,
                n_train=2000, n_val=500, n_test=500)
        STATE['A'] = loaders
        print(f"  [phase A data] ready in {time.time()-t0:.0f}s")
    return STATE['A']

def cfg_A():
    c = CfgAB()
    if SMOKE:
        c.epochs = 1; c.max_train_batch = 2
        c.max_val_batch = 2; c.patience = 1
    return c

def job_A(seed, run_idx, block, cfgname):
    tl, vl, tel = get_data_A()
    set_run_seed(seed, run_idx)
    d = dict(nm=80, dm=128, du=512, ns=4, nc=4, sr=16000)
    d.update(JOB_KW_A[block][cfgname])
    model = FME_AB(**d)
    prm = model.nparams()
    tr = TrainerAB(model, cfg_A(), tl, vl, DEVICE)
    r = tr.run(verbose=False)
    del tr; model.cpu(); del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'A', 'seed': seed, 'block': block, 'config': cfgname,
            'UA': float(r['UA']), 'F1': float(r['F1']), 'RTF': float(r['RTF']),
            'time': float(r['time']), 'epochs': int(r['epochs']),
            'params': int(prm), 'best_ua': float(r['best_ua'])}

def jobs_A():
    out = []
    for seed in SEEDS_A:
        for idx, (block, cfgname) in enumerate(ORDER_A):
            out.append((
                f"A|{seed}|{block}|{cfgname}",
                lambda s=seed, i=idx, b=block, c=cfgname:
                    job_A(s, i, b, c)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE B — Hurst regularizer ON vs OFF
# ═══════════════════════════════════════════════════════════════════════════
class TrainerHurst(nn.Module):
    def __init__(self, model, freg, captured, lam, cfg, tl, vl, dev, ns):
        super().__init__()
        self.m = model.to(dev); self.c = cfg; self.d = dev
        self.tl = tl; self.vl = vl
        self.freg = freg.to(dev); self.captured = captured
        self.lam = lam; self.ns = ns
        self.opt = torch.optim.AdamW(
            model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        steps = cfg.epochs * min(len(tl), cfg.max_train_batch)
        self.sched = torch.optim.lr_scheduler.OneCycleLR(
            self.opt, max_lr=cfg.lr,
            total_steps=max(steps, 1), pct_start=0.15)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.ua_fn   = NS_AB['ua_score']
        self.best = 0; self.wait = 0
        self.hurst_log = []
        self.stride_rates = (1, 2, 4, 8, 16)[:ns]

    def _fractal(self, out):
        seqs = [self.captured[i] for i in range(self.ns)
                if i in self.captured]
        if len(seqs) < 2:
            z = out['z_unified']
            seqs = [z, z]
        feats = out['scale_features']
        if not isinstance(feats, (list, tuple)):
            feats = [feats]
        loss, metrics = self.freg(
            seqs, list(feats)[:self.ns],
            self.stride_rates[:len(seqs)])
        return loss, metrics

    def train_ep(self):
        self.m.train(); P = []; T = []; ep_h = []
        for i, b in enumerate(self.tl):
            if i >= self.c.max_train_batch: break
            w = b['waveform'].to(self.d, non_blocking=True)
            e = b['emotion'].to(self.d, non_blocking=True)
            self.opt.zero_grad(set_to_none=True)
            try:
                out = self.m(w)
                l = self.loss_fn(out['predictions']['emotion_logits'], e)
                fl, fm = self._fractal(out)
                if self.lam > 0:
                    l = l + self.lam * fl
                if not torch.isfinite(l): continue
                l.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.m.parameters(), self.c.clip)
                self.opt.step(); self.sched.step()
                P.extend(out['predictions']['emotion_logits']
                         .argmax(-1).cpu().numpy())
                T.extend(e.cpu().numpy())
                ep_h.append(float(fm['hurst_mean']))
            except RuntimeError:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        self.hurst_log.append(
            float(np.mean(ep_h)) if ep_h else float('nan'))
        return self.ua_fn(P, T) if T else 0

    @torch.no_grad()
    def eval(self):
        self.m.eval(); P = []; T = []
        f1_score = NS_AB['f1_score']
        for i, b in enumerate(self.vl):
            if i >= self.c.max_val_batch: break
            w = b['waveform'].to(self.d, non_blocking=True)
            e = b['emotion'].to(self.d, non_blocking=True)
            try:
                p = self.m(w)['predictions']['emotion_logits'].argmax(-1)
                P.extend(p.cpu().numpy())
                T.extend(e.cpu().numpy())
            except Exception:
                pass
        ua = self.ua_fn(P, T) if T else 0
        f1 = (f1_score(T, P, average='macro', zero_division=0) * 100
              if T else 0)
        return ua, f1

    def run(self):
        t0 = time.time(); ep = 0
        for ep in range(self.c.epochs):
            tua = self.train_ep()
            vua, vf1 = self.eval()
            if vua > self.best: self.best = vua; self.wait = 0
            else: self.wait += 1
            print(f"      [hurst-run] ep{ep+1}: trainUA={tua:.1f} "
                  f"valUA={vua:.1f} "
                  f"H_mean={self.hurst_log[-1]:.3f}")
            if self.wait >= self.c.patience: break
        fua, ff1 = self.eval()
        r = rtf_ab(self.m, self.vl, self.d)
        return {'UA': fua, 'F1': ff1, 'RTF': r,
                'time': time.time() - t0, 'epochs': ep + 1,
                'best_ua': self.best,
                'hurst_per_epoch': self.hurst_log,
                'hurst_final': (self.hurst_log[-1]
                                if self.hurst_log else float('nan'))}

def job_B(lam, seed, run_idx):
    tl, vl, tel = get_data_A()
    set_run_seed(seed, run_idx)
    ns = 4
    model = FME_AB(nm=80, dm=128, du=512, ns=ns, nc=4, sr=16000)
    freg  = FractalReg(target_hurst=0.7, num_scales=ns, temperature=10.0)
    captured = {}
    hooks = []
    for i, p in enumerate(model.paths):
        hooks.append(p.pool.register_forward_pre_hook(
            lambda mod, inp, i=i:
                captured.__setitem__(i, inp[0])))
    prm = model.nparams()
    tr = TrainerHurst(model, freg, captured, lam,
                      cfg_A(), tl, vl, DEVICE, ns)
    r = tr.run()
    for h in hooks:
        h.remove()
    del tr; model.cpu(); del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'B', 'seed': seed, 'lambda_fractal': lam,
            'UA': float(r['UA']), 'F1': float(r['F1']),
            'RTF': float(r['RTF']), 'time': float(r['time']),
            'epochs': int(r['epochs']), 'params': int(prm),
            'hurst_per_epoch': [float(x) for x in r['hurst_per_epoch']],
            'hurst_final': float(r['hurst_final'])}

def jobs_B():
    out = []
    for lam in (1.0, 0.0):
        for j, seed in enumerate(SEEDS_B):
            out.append((
                f"B|lam{lam}|{seed}",
                lambda l=lam, s=seed, i=j: job_B(l, s, i)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE C — Table-2 quick config, multi-seed
# ═══════════════════════════════════════════════════════════════════════════
def job_C(seed):
    if SMOKE:
        kw = dict(num_epochs=1, batch_size=4, num_scales=3,
                  d_model=64, d_unified=256,
                  num_train=32, num_val=16, num_test=16)
    else:
        kw = dict(num_epochs=10, batch_size=12, num_scales=3,
                  d_model=64, d_unified=256,
                  num_train=800, num_val=160, num_test=160)
    res = NS_QP['run_experiment'](
        exp_name=f"fme_quick_rev_s{seed}", seed=seed,
        run_ablation_study=False, verbose=False, **kw)
    ser = NS_QP.get('to_serializable', lambda x: x)
    out = {'phase': 'C', 'seed': seed}
    tm  = res.get('test_metrics') or res.get('metrics') or {}
    out['test_metrics'] = jclean(ser(tm))
    eff = res.get('efficiency') or {}
    out['efficiency']   = jclean(ser(eff))
    out['params']       = int(res.get('params', 0))
    bench = res.get('benchmark_table')
    if bench:
        out['benchmark_table_rows'] = jclean(ser(bench))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def jobs_C():
    return [(f"C|{s}", lambda s=s: job_C(s)) for s in SEEDS_C]

# ═══════════════════════════════════════════════════════════════════════════
# PHASE D — unified RTF measurement, both protocols
# ═══════════════════════════════════════════════════════════════════════════
def _model_quick3():
    cfg   = NS_QP['Config'](exp_name='rtf_probe',
                            num_scales=3, d_model=64, d_unified=256)
    model = NS_QP['FractalMambaEmbedding'](cfg)
    return model, cfg

def _model_abl4():
    return FME_AB(nm=80, dm=128, du=512, ns=4, nc=4, sr=16000), None

_D_IDX = {('quick3', 'single_utterance'): 0,
          ('quick3', 'batched'):           1,
          ('abl4',   'single_utterance'): 2,
          ('abl4',   'batched'):           3}

def job_D(variant, protocol):
    set_run_seed(42, 900 + _D_IDX[(variant, protocol)])
    if variant == 'quick3':
        model, cfg = _model_quick3()
    else:
        model, cfg = _model_abl4()
    model = model.to(DEVICE); model.eval()
    prm = sum(p.numel() for p in model.parameters())
    durations = [1] if SMOKE else [1, 5, 10, 30, 60]
    out = {'phase': 'D', 'variant': variant, 'protocol': protocol,
           'params': int(prm), 'device': str(DEVICE),
           'gpu': (torch.cuda.get_device_name(0)
                   if DEVICE.type == 'cuda' else 'cpu')}
    try:
        if protocol == 'single_utterance':
            if cfg is None:
                cfg = NS_QP['Config'](exp_name='rtf_probe')
            ev  = NS_QP['Evaluator'](model, cfg)
            eff = ev.benchmark_efficiency(durations=durations)
            out['latency'] = jclean(
                {str(k): {'time_ms': float(v['time_ms']),
                          'rtf':     float(v['rtf'])}
                 for k, v in eff.items()})
        else:
            tl, vl, tel = get_data_A()
            r_batch = rtf_ab(model, vl, DEVICE,
                             nb=(1 if SMOKE else 3))
            out['rtf_batched'] = float(r_batch)
            out['batch_size']  = (8 if SMOKE else 48)
    finally:
        model.cpu(); del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return out

def jobs_D():
    out = []
    for variant in ('quick3', 'abl4'):
        for protocol in ('single_utterance', 'batched'):
            out.append((
                f"D|{variant}|{protocol}",
                lambda v=variant, p=protocol: job_D(v, p)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE E — FME on REAL corpora
# ═══════════════════════════════════════════════════════════════════════════
CORPUS_SPECS = {
    'cremad': {
        'kaggle':      'ejlok1/cremad',
        'urls':        [],
        'mount_globs': ['/kaggle/input/cremad/AudioWAV',
                        '/kaggle/input/*/AudioWAV'],
    },
    'ravdess': {
        'kaggle': 'uwrfkaggler/ravdess-emotional-speech-audio',
        'urls':   ['https://zenodo.org/record/1188976/files/'
                   'Audio_Speech_Actors_01-24.zip'],
        'mount_globs': [
            '/kaggle/input/ravdess-emotional-speech-audio',
            '/kaggle/input/*/audio_speech_actors_01-24'],
    },
}
EMO4    = {'NEU': 0, 'HAP': 1, 'SAD': 2, 'ANG': 3}
RAV_EMO = {'01': 'NEU', '03': 'HAP', '04': 'SAD', '05': 'ANG'}

def _glob_first(patterns):
    import glob as _glob
    for pat in patterns:
        hits = sorted(_glob.glob(pat))
        if hits:
            return hits[0]
    return None

def _download_corpus(name):
    spec = CORPUS_SPECS[name]
    dest = Path('./data') / name
    dest.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.check_call(
            ['kaggle', 'datasets', 'download', '-d', spec['kaggle'],
             '-p', str(dest), '--unzip'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            timeout=1800)
        print(f"  [phase E] {name}: downloaded via kaggle CLI")
    except Exception as e:
        print(f"  [phase E] {name}: kaggle CLI failed ({e})")
        for url in spec['urls']:
            try:
                zp = dest / url.split('/')[-1]
                if not zp.exists():
                    print(f"  [phase E] downloading {url}")
                    urllib.request.urlretrieve(url, zp)
                with zipfile.ZipFile(zp) as z:
                    z.extractall(dest)
                print(f"  [phase E] {name}: downloaded via URL")
                break
            except Exception as e2:
                print(f"  [phase E] URL failed: {e2}")
    return dest

def _find_wavs(name):
    import glob as _glob
    roots = []
    dest  = Path('./data') / name
    roots += ([dest] +
              [Path(p) for p in
               [g.replace('*', '*')
                for g in CORPUS_SPECS[name]['mount_globs']]])
    pats = []
    for r in roots:
        pats += [f"{r}/**/*.wav", f"{r}/**/*.WAV"]
    seen = {}
    for pat in pats:
        for f in _glob.glob(pat, recursive=True):
            seen[os.path.basename(f)] = f
    return seen

def _parse_cremad(fname):
    p = fname.split('.')[0].split('_')
    if len(p) < 4: return None
    actor, emo = p[0], p[2].upper()
    if emo not in EMO4: return None
    return actor, EMO4[emo]

def _parse_ravdess(fname):
    p = fname.split('.')[0].split('-')
    if len(p) < 7 or p[1] != '01': return None
    emo = RAV_EMO.get(p[2])
    if emo is None: return None
    return p[6], EMO4[emo]

def get_corpus(name):
    if f"corp_{name}" in STATE:
        return STATE[f"corp_{name}"]
    parser = _parse_cremad if name == 'cremad' else _parse_ravdess
    files  = _find_wavs(name)
    if len(files) < 100:
        _download_corpus(name)
        files = _find_wavs(name)
    items, skipped = [], 0
    for fname, path in sorted(files.items()):
        r = parser(fname)
        if r is None: skipped += 1; continue
        items.append((path, r[0], r[1]))
    if not items:
        raise RuntimeError(
            f"corpus {name}: no usable wav files found "
            f"(skipped {skipped})")
    actors = sorted(set(a for _, a, _ in items))
    rng  = np.random.RandomState(42)
    perm = list(rng.permutation(len(actors)))
    n    = len(perm)
    tr_a = set(actors[perm[i]] for i in range(int(0.70 * n)))
    va_a = set(actors[perm[i]]
               for i in range(int(0.70 * n), int(0.85 * n)))
    splits = {'train': [], 'val': [], 'test': []}
    for path, actor, emo in items:
        s = ('train' if actor in tr_a
             else ('val' if actor in va_a else 'test'))
        splits[s].append((path, emo))
    info = {'name': name,
            'n_files_total': len(files),
            'n_used': len(items),
            'skipped_non4class_or_bad': skipped,
            'split_sizes': {k: len(v) for k, v in splits.items()},
            'class_dist': {k: dict(Counter(e for _, e in v))
                           for k, v in splits.items()},
            'n_actors': len(actors)}
    STATE[f"corp_{name}"] = (splits, info)
    print(f"  [phase E] corpus {name}: {info}")
    return STATE[f"corp_{name}"]

class RealAudioDataset(Dataset):
    def __init__(self, items, max_sec=8.0, sr=16000):
        self.items   = items; self.sr = sr
        self.max_len = int(max_sec * sr)
        self.bad     = []

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        import soundfile as sf
        wav, sr = sf.read(path, dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != self.sr:
            import torchaudio.functional as AF
            wav = AF.resample(
                torch.from_numpy(wav), sr, self.sr).numpy()
        return wav

    def __getitem__(self, idx):
        path, emo = self.items[idx]
        try:
            wav = self._load(path)
        except Exception:
            wav = np.zeros(self.max_len, dtype=np.float32)
            emo = -100
            self.bad.append(path)
        w = torch.from_numpy(np.asarray(wav, dtype=np.float32))
        if w.shape[0] > self.max_len:
            st = torch.randint(0, w.shape[0] - self.max_len, (1,)).item()
            w  = w[st:st + self.max_len]
        elif w.shape[0] < self.max_len:
            w = F.pad(w, (0, self.max_len - w.shape[0]))
        return {'waveform': w, 'emotion': int(emo)}

def _real_loaders(name):
    key = f"loaders_{name}"
    if key in STATE:
        return STATE[key]
    splits, info = get_corpus(name)
    bs = 8 if SMOKE else 32
    ds = {k: RealAudioDataset(v) for k, v in splits.items()}
    loaders = {
        k: DataLoader(d, batch_size=bs, shuffle=(k == 'train'),
                      num_workers=0,
                      pin_memory=(DEVICE.type == 'cuda'),
                      drop_last=False)
        for k, d in ds.items()}
    STATE[key] = (loaders, info)
    return STATE[key]

def eval_full(model, loader):
    from sklearn.metrics import confusion_matrix
    model.eval(); P = []; T = []
    with torch.no_grad():
        for b in loader:
            w = b['waveform'].to(DEVICE)
            e = b['emotion']
            ok = e >= 0
            if ok.sum() == 0: continue
            p = (model(w.to(DEVICE))['predictions']['emotion_logits']
                 .argmax(-1).cpu())
            P.extend(p[ok].numpy()); T.extend(e[ok].numpy())
    P = np.array(P); T = np.array(T)
    acc = float((P == T).mean() * 100)
    ua  = float(NS_AB['ua_score'](P, T))
    f1  = float(NS_AB['f1_score'](
        T, P, average='macro', zero_division=0) * 100)
    rec = NS_AB['recall_score'](
        T, P, average=None, labels=[0, 1, 2, 3], zero_division=0)
    cm  = confusion_matrix(T, P, labels=[0, 1, 2, 3])
    return {'acc': acc, 'UA': ua, 'macroF1': f1, 'n': int(len(T)),
            'per_class_recall': {n2: float(x) for n2, x in
                                 zip(['NEU','HAP','SAD','ANG'], rec)},
            'confusion': cm.tolist()}

def job_E(corpus, seed):
    (loaders, info) = _real_loaders(corpus)
    set_run_seed(seed, 700 + SEEDS_E.index(seed))
    model = FME_AB(nm=80, dm=128, du=512,
                   ns=4, nc=4, sr=16000).to(DEVICE)
    prm = model.nparams()
    c   = CfgAB()
    if SMOKE:
        c.epochs = 1; c.max_train_batch = 2
        c.max_val_batch = 2; c.patience = 1
    else:
        c.epochs = 25; c.patience = 6
        c.max_train_batch = 300; c.max_val_batch = 100
    tr     = TrainerAB(model, c, loaders['train'], loaders['val'], DEVICE)
    r      = tr.run(verbose=False)
    test_m = eval_full(model, loaders['test'])
    val_m  = eval_full(model, loaders['val'])
    rtf_b  = rtf_ab(model, loaders['test'], DEVICE,
                    nb=(1 if SMOKE else 3))
    model.cpu(); del model, tr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'E', 'corpus': corpus, 'seed': seed,
            'params': int(prm), 'test': test_m, 'val': val_m,
            'val_protocol_UA': float(r['UA']),
            'val_protocol_F1': float(r['F1']),
            'rtf_batched': float(rtf_b),
            'time': float(r['time']), 'corpus_info': info}

def jobs_E():
    out = []
    for corpus in REAL_CORPORA:
        for seed in SEEDS_E:
            out.append((
                f"E|{corpus}|{seed}",
                lambda c=corpus, s=seed: job_E(c, s)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE F — SSL baselines fine-tuned
# ═══════════════════════════════════════════════════════════════════════════
class SSLClassifier(nn.Module):
    def __init__(self, encoder, hidden, ncls=4, pool='mean'):
        super().__init__()
        self.enc  = encoder
        self.head = nn.Linear(hidden, ncls)
        self.pool = pool
    def forward(self, wav):
        x = wav
        x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)
        h = self.enc(input_values=x).last_hidden_state
        z = h.mean(1) if self.pool == 'mean' else h[:, -1]
        return {'predictions': {'emotion_logits': self.head(z)}}

def _ssl_loader(name, batch_size):
    (loaders, info) = _real_loaders(name)
    return loaders, info

def job_F_finetune(mkey, corpus, seed):
    _pip_install(['transformers'])
    from transformers import AutoModel
    (loaders, info) = _ssl_loader(corpus, 8)
    set_run_seed(seed, 800 + SEEDS_F.index(seed))
    enc    = AutoModel.from_pretrained(SSL_MODELS[mkey])
    hidden = enc.config.hidden_size
    model  = SSLClassifier(enc, hidden).to(DEVICE)
    prm    = sum(p.numel() for p in model.parameters())
    epochs = 1 if SMOKE else 8
    lr     = 1e-5
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps  = epochs * max(len(loaders['train']), 1)
    sched  = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max(steps, 1), pct_start=0.1)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    t0 = time.time()
    for ep in range(epochs):
        model.train(); n_seen = 0
        for b in loaders['train']:
            w = b['waveform'].to(DEVICE)
            e = b['emotion'].to(DEVICE)
            ok = e >= 0
            if ok.sum() == 0: continue
            opt.zero_grad(set_to_none=True)
            out = model(w)['predictions']['emotion_logits']
            l   = loss_fn(out, e)
            if not torch.isfinite(l): continue
            l.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            n_seen += 1
            if SMOKE and n_seen >= 2: break
        print(f"      [F {mkey}/{corpus}] epoch {ep+1}/{epochs} done "
              f"({(time.time()-t0)/60:.1f} min)")
    test_m = eval_full(model, loaders['test'])
    model.cpu(); del model, enc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'F', 'kind': 'finetune', 'model': mkey,
            'hf_id': SSL_MODELS[mkey], 'corpus': corpus, 'seed': seed,
            'params': int(prm), 'epochs': epochs,
            'time': time.time() - t0, 'test': test_m,
            'corpus_info': info}

def job_F_rtf(mkey):
    _pip_install(['transformers'])
    from transformers import AutoModel
    set_run_seed(42, 850)
    enc   = AutoModel.from_pretrained(SSL_MODELS[mkey])
    model = SSLClassifier(enc, enc.config.hidden_size).to(DEVICE).eval()
    prm   = sum(p.numel() for p in model.parameters())
    out   = {'phase': 'F', 'kind': 'rtf', 'model': mkey,
             'hf_id': SSL_MODELS[mkey], 'params': int(prm),
             'device': str(DEVICE),
             'gpu': (torch.cuda.get_device_name(0)
                     if DEVICE.type == 'cuda' else 'cpu')}
    tl, vl, tel = get_data_A()
    class _Wrap(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, wav):
            w = (wav[:, :128000] if wav.shape[1] > 128000
                 else F.pad(wav, (0, 128000 - wav.shape[1])))
            return self.m(w)
    wrap = _Wrap(model)
    out['rtf_batched_8s'] = float(
        rtf_ab(wrap, vl, DEVICE, nb=(1 if SMOKE else 3)))
    lat = {}
    for dur in ([1] if SMOKE else [1, 5, 10]):
        wave = torch.randn(1, dur * 16000, device=DEVICE)
        with torch.no_grad():
            for _ in range(2): wrap(wave)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            ts = []
            for _ in range(5 if SMOKE else 10):
                t0 = time.perf_counter()
                wrap(wave)
                if DEVICE.type == 'cuda': torch.cuda.synchronize()
                ts.append(time.perf_counter() - t0)
        lat[f'{dur}s'] = {'time_ms': float(np.mean(ts) * 1000),
                          'rtf':     float(np.mean(ts) / dur)}
    out['latency_single_utterance'] = lat
    model.cpu(); del model, enc, wrap
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def job_F_e2v(corpus):
    if not TRY_EMOTION2VEC:
        return {'phase': 'F', 'kind': 'emotion2vec', 'corpus': corpus,
                'status': 'SKIPPED', 'reason': 'TRY_EMOTION2VEC=False'}
    try:
        _pip_install(['funasr', 'modelscope'])
        from funasr import AutoModel as FAutoModel
        from sklearn.linear_model import LogisticRegression
        splits, info = get_corpus(corpus)
        e2v = FAutoModel(model="iic/emotion2vec_plus_base",
                         device=str(DEVICE))
        def embed(items, limit=None):
            X = []; Y = []
            lst = items if limit is None else items[:limit]
            for path, emo in lst:
                if emo < 0: continue
                try:
                    r = e2v.generate(path, granularity="utterance",
                                     extract_embedding=True)
                    v = np.array(r[0]['feats'], dtype=np.float32)
                    X.append(v); Y.append(emo)
                except Exception:
                    continue
            return np.array(X), np.array(Y)
        lim = 32 if SMOKE else None
        Xtr, Ytr = embed(splits['train'], lim)
        Xte, Yte = embed(splits['test'],  lim)
        clf = LogisticRegression(max_iter=2000).fit(Xtr, Ytr)
        P   = clf.predict(Xte)
        out = {'phase': 'F', 'kind': 'emotion2vec_probe',
               'corpus': corpus,
               'protocol': ('frozen embeddings + logistic regression '
                            '(fine-tuning not supported by the official '
                            'runtime)'),
               'dim': int(Xtr.shape[1]),
               'n_train': int(len(Ytr)), 'n_test': int(len(Yte)),
               'test': {
                   'acc':     float((P == Yte).mean() * 100),
                   'UA':      float(NS_AB['ua_score'](P, Yte)),
                   'macroF1': float(NS_AB['f1_score'](
                       Yte, P, average='macro', zero_division=0) * 100)}}
        del e2v; gc.collect()
        return out
    except Exception as e:
        return {'phase': 'F', 'kind': 'emotion2vec_probe',
                'corpus': corpus, 'status': 'SKIPPED',
                'reason': f'{type(e).__name__}: {e}'}

def jobs_F():
    out = []
    for corpus in REAL_CORPORA:
        for mkey in SSL_MODELS:
            for seed in SEEDS_F:
                out.append((
                    f"F|ft|{mkey}|{corpus}|{seed}",
                    lambda m=mkey, c=corpus, s=seed:
                        job_F_finetune(m, c, s)))
    for corpus in REAL_CORPORA:
        out.append((
            f"F|e2v|{corpus}",
            lambda c=corpus: job_F_e2v(c)))
    for mkey in SSL_MODELS:
        out.append((
            f"F|rtf|{mkey}",
            lambda m=mkey: job_F_rtf(m)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE G — long-form utterances
# ═══════════════════════════════════════════════════════════════════════════
G_SPEC = {
    20: dict(min_sec=18.0, max_sec=22.0,
             n_train=300, n_val=96, n_test=96, batch=8),
    60: dict(min_sec=55.0, max_sec=61.0,
             n_train=100, n_val=48, n_test=48, batch=4),
}

def get_data_G(dur):
    key = f"G_{dur}"
    if key in STATE:
        return STATE[key]
    spec = G_SPEC[dur]
    t0   = time.time()
    if SMOKE:
        spec = dict(spec)
        spec.update(n_train=4, n_val=4, n_test=4,
                    min_sec=2.0, max_sec=3.0, batch=2)
    mk = lambda split, n, sd: NS_AB['HardEmotionDataset'](
        split, max_sec=spec['max_sec'], min_sec=spec['min_sec'],
        max_samples=n, difficulty='hard', seed=sd)
    tds = mk('train',      spec['n_train'], 42)
    vds = mk('validation', spec['n_val'],  123)
    eds = mk('test',       spec['n_test'], 456)
    kw  = dict(batch_size=spec['batch'], num_workers=0,
               pin_memory=(DEVICE.type == 'cuda'), drop_last=False)
    loaders = (DataLoader(tds, shuffle=True,  **kw),
               DataLoader(vds, shuffle=False, **kw),
               DataLoader(eds, shuffle=False, **kw))
    STATE[key] = loaders
    print(f"  [phase G] {dur}s data ready in {time.time()-t0:.0f}s")
    return loaders

def job_G(dur, ns, seed):
    tl, vl, el = get_data_G(dur)
    set_run_seed(seed, 600 + ns * 10 + dur)
    model = FME_AB(nm=80, dm=128, du=512,
                   ns=ns, nc=4, sr=16000).to(DEVICE)
    prm = model.nparams()
    c   = CfgAB()
    if SMOKE:
        c.epochs = 1; c.max_train_batch = 2
        c.max_val_batch = 2; c.patience = 1
    else:
        c.epochs = 5; c.patience = 3
        c.max_train_batch = 40; c.max_val_batch = 20
    tr     = TrainerAB(model, c, tl, vl, DEVICE)
    r      = tr.run(verbose=False)
    test_m = eval_full(model, el)
    model.cpu(); del model, tr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'G', 'duration_s': dur, 'num_scales': ns,
            'seed': seed, 'params': int(prm),
            'UA_val': float(r['UA']), 'F1_val': float(r['F1']),
            'RTF': float(r['RTF']), 'time': float(r['time']),
            'test': test_m}

def jobs_G():
    out = []
    for dur in (20, 60):
        for ns in (1, 4, 5):
            for seed in SEEDS_G:
                if SMOKE and not (dur == 20 and ns == 4
                                  and seed == SEEDS_G[0]):
                    continue
                out.append((
                    f"G|{dur}s|S{ns}|{seed}",
                    lambda d=dur, n=ns, s=seed: job_G(d, n, s)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE H — CTC lexical decoding
# ═══════════════════════════════════════════════════════════════════════════
OPENSLR = {
    'dev-clean':  ('https://www.openslr.org/resources/12/'
                   'dev-clean.tar.gz'),
    'test-clean': ('https://www.openslr.org/resources/12/'
                   'test-clean.tar.gz'),
}
CTC_VOCAB  = list("abcdefghijklmnopqrstuvwxyz' ")
CTC_STATE  = CACHE_DIR / 'ctc_model.pt'

def _edit_distance(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1,
                           prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def _wer_cer(hyp, ref):
    hw, rw = hyp.split(), ref.split()
    hc = [c for c in hyp if c != ' ']
    rc = [c for c in ref if c != ' ']
    wer = _edit_distance(hw, rw) / max(len(rw), 1) * 100
    cer = _edit_distance(hc, rc) / max(len(rc), 1) * 100
    return wer, cer

class CTCNet(nn.Module):
    def __init__(self):
        super().__init__()
        import torchaudio.transforms as T
        self.mel = nn.Sequential(
            T.MelSpectrogram(sample_rate=16000, n_fft=512,
                             hop_length=160, n_mels=80,
                             f_min=20, f_max=8000),
            T.AmplitudeToDB(stype='power', top_db=80))
        self.fe   = NS_AB['Frontend'](80, 256, 1, 2, True)
        self.blks = nn.ModuleList([
            NS_AB['BiMambaBlock'](256, 0.1) for _ in range(3)])
        self.norm = NS_AB['RMSNorm'](256)
        self.head = nn.Linear(256, len(CTC_VOCAB) + 1)

    def forward(self, wav):
        w = ((wav - wav.mean(-1, keepdim=True)) /
             (wav.std(-1, keepdim=True) + 1e-8))
        m = self.mel(w)
        m = (m - m.mean()) / (m.std() + 1e-8)
        x = m.transpose(1, 2).clamp(-10, 10)
        x = self.fe(x)
        for b in self.blks: x = b(x)
        return torch.log_softmax(self.head(self.norm(x)), dim=-1)

def _ls_manifest(root):
    import glob as _glob
    utts = []
    for tr in sorted(_glob.glob(f"{root}/**/*.trans.txt",
                                recursive=True)):
        base = os.path.dirname(tr)
        for line in open(tr):
            parts = line.strip().split(' ', 1)
            if len(parts) != 2: continue
            uid, text = parts
            flac = f"{base}/{uid}.flac"
            if os.path.exists(flac):
                utts.append((flac, text.lower().replace('-', ' ')))
    return utts

def job_H_prep():
    if SMOKE:
        return {'phase': 'H', 'kind': 'prep',
                'status': 'SKIPPED', 'reason': 'SMOKE mode'}
    dest = Path('./data/librispeech')
    dest.mkdir(parents=True, exist_ok=True)
    for name, url in OPENSLR.items():
        marker = dest / f'.{name}.done'
        if marker.exists(): continue
        tgz = dest / f'{name}.tar.gz'
        try:
            if not tgz.exists():
                print(f"  [phase H] downloading {url} (~340 MB)")
                t0 = time.time()
                urllib.request.urlretrieve(url, tgz)
                print(f"  [phase H] downloaded in "
                      f"{(time.time()-t0)/60:.1f} min")
            with tarfile.open(tgz) as t:
                t.extractall(dest)
            marker.write_text('ok')
        except Exception as e:
            return {'phase': 'H', 'kind': 'prep',
                    'status': 'SKIPPED',
                    'reason': (f'download/extract failed: '
                               f'{type(e).__name__}: {e}')}
    all_utts = _ls_manifest(dest)
    n_dev  = len([u for u in all_utts if '/dev-clean/'  in u[0]])
    n_test = len([u for u in all_utts if '/test-clean/' in u[0]])
    if n_dev < 100 or n_test < 100:
        return {'phase': 'H', 'kind': 'prep', 'status': 'SKIPPED',
                'reason': (f'manifest too small '
                           f'(dev={n_dev}, test={n_test})')}
    return {'phase': 'H', 'kind': 'prep', 'status': 'OK',
            'n_dev_clean': n_dev, 'n_test_clean': n_test,
            'vocab': len(CTC_VOCAB) + 1}

def _h_utts(which):
    key = f"H_utts_{which}"
    if key not in STATE:
        all_utts = _ls_manifest(Path('./data/librispeech'))
        STATE['H_utts_dev']  = [u for u in all_utts
                                 if '/dev-clean/'  in u[0]]
        STATE['H_utts_test'] = [u for u in all_utts
                                 if '/test-clean/' in u[0]]
        random.Random(42).shuffle(STATE['H_utts_dev'])
        random.Random(42).shuffle(STATE['H_utts_test'])
    return STATE[key]

class LSCTCDataset(Dataset):
    def __init__(self, utts, max_sec=10.0, sr=16000):
        self.utts    = utts
        self.max_len = int(max_sec * sr)
    def __len__(self): return len(self.utts)
    def __getitem__(self, i):
        import soundfile as sf
        path, text = self.utts[i]
        try:
            wav, sr = sf.read(path, dtype='float32')
        except Exception:
            wav = np.zeros(self.max_len, dtype=np.float32)
        w = torch.from_numpy(wav[:self.max_len])
        if w.shape[0] < self.max_len:
            w = F.pad(w, (0, self.max_len - w.shape[0]))
        tgt = torch.tensor(
            [CTC_VOCAB.index(c) for c in text if c in CTC_VOCAB],
            dtype=torch.long)
        return w, tgt, text

def _ctc_collate(batch):
    ws    = torch.stack([b[0] for b in batch])
    tgts  = [b[1] for b in batch]
    texts = [b[2] for b in batch]
    return ws, tgts, texts

def _greedy_decode(lp):
    ids = lp.argmax(-1).tolist()
    out = []; prev = -1
    for x in ids:
        if x != prev and x != len(CTC_VOCAB):
            out.append(CTC_VOCAB[x])
        prev = x
    return ''.join(out)

def job_H_epoch(ep):
    prep = CACHE.get('H|prep', {})
    if prep.get('status') != 'OK':
        return {'phase': 'H', 'kind': f'epoch{ep}',
                'status': 'SKIPPED',
                'reason': (f"prep status: "
                           f"{prep.get('status', 'not run')}")}
    set_run_seed(42, 950 + ep)
    model     = CTCNet().to(DEVICE)
    opt_state = None
    if ep > 1 and CTC_STATE.exists():
        try:
            ck = torch.load(CTC_STATE, map_location=DEVICE)
            model.load_state_dict(ck['model'])
            opt_state = ck.get('opt')
        except Exception as e:
            print(f"  [phase H] checkpoint load failed ({e}); "
                  f"restarting model")
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    if opt_state:
        try: opt.load_state_dict(opt_state)
        except Exception: pass
    dev = _h_utts('H_utts_dev')
    ds  = LSCTCDataset(dev)
    dl  = DataLoader(ds, batch_size=(2 if SMOKE else 8),
                     shuffle=True, num_workers=0,
                     collate_fn=_ctc_collate)
    model.train(); t0 = time.time()
    tot = 0.0; nb = 0
    max_batches = 4 if SMOKE else 10**9
    for i, (w, tgts, _texts) in enumerate(dl):
        if i >= max_batches: break
        w  = w.to(DEVICE)
        lp = model(w).transpose(0, 1)
        Tlen = torch.full((lp.shape[1],), lp.shape[0],
                          dtype=torch.long, device=DEVICE)
        keep = [t for t in tgts if len(t) > 0]
        if not keep: continue
        tgt_cat = torch.cat(tgts).to(DEVICE)
        tlen    = torch.tensor([len(t) for t in tgts],
                               dtype=torch.long, device=DEVICE)
        if (tlen == 0).any(): continue
        loss = F.ctc_loss(lp, tgt_cat, Tlen, tlen,
                          blank=len(CTC_VOCAB), zero_infinity=True)
        if not torch.isfinite(loss): continue
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        tot += float(loss); nb += 1
        if (i + 1) % 200 == 0:
            print(f"      [H ep{ep}] batch {i+1} "
                  f"loss={tot/max(nb,1):.3f} "
                  f"({(time.time()-t0)/60:.1f} min)")
    test = _h_utts('H_utts_test')[:(16 if SMOKE else 500)]
    tds  = LSCTCDataset(test)
    tdl  = DataLoader(tds, batch_size=(2 if SMOKE else 8),
                      shuffle=False, num_workers=0,
                      collate_fn=_ctc_collate)
    model.eval(); wers = []; cers = []; samples = []
    with torch.no_grad():
        for w, _tgts, texts in tdl:
            lp = model(w.to(DEVICE))
            for j in range(lp.shape[0]):
                hyp = _greedy_decode(lp[j])
                wj, cj = _wer_cer(hyp, texts[j])
                wers.append(wj); cers.append(cj)
                if len(samples) < 5:
                    samples.append({'ref': texts[j][:80],
                                    'hyp': hyp[:80]})
    torch.save({'model': model.state_dict(),
                'opt':   opt.state_dict()}, CTC_STATE)
    model.cpu(); del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'phase': 'H', 'kind': f'epoch{ep}',
            'train_loss': tot / max(nb, 1), 'batches': nb,
            'time_min': (time.time() - t0) / 60,
            'eval_n': len(wers),
            'WER': float(np.mean(wers)),
            'CER': float(np.mean(cers)),
            'decode_samples': samples}

def jobs_H():
    out = [('H|prep', job_H_prep)]
    for ep in range(1, CTC_EPOCHS + 1):
        out.append((f"H|ep{ep}", lambda e=ep: job_H_epoch(e)))
    return out

# ═══════════════════════════════════════════════════════════════════════════
# PHASE BUILDER MAP
# ═══════════════════════════════════════════════════════════════════════════
PHASE_BUILDERS = {
    'A': jobs_A, 'B': jobs_B, 'C': jobs_C, 'D': jobs_D,
    'E': jobs_E, 'F': jobs_F, 'G': jobs_G, 'H': jobs_H,
}

# ── SMOKE adjustments ────────────────────────────────────────────────────────
if SMOKE:
    SEEDS_A = [0, 1]; SEEDS_B = [0]; SEEDS_C = [0]; SEEDS_G = [0]
    if CURRENT_PHASE == 'C':
        _be = NS_QP['Evaluator'].benchmark_efficiency
        NS_QP['Evaluator'].benchmark_efficiency = (
            lambda self, durations=None: _be(self, [1]))

# ═══════════════════════════════════════════════════════════════════════════
# BUILD JOB LIST FOR CURRENT PHASE ONLY
# ═══════════════════════════════════════════════════════════════════════════
ALL_JOBS_PHASE = PHASE_BUILDERS[CURRENT_PHASE]()
PENDING = [(jid, fn) for jid, fn in ALL_JOBS_PHASE if jid not in CACHE]

print(f"\n  Phase {CURRENT_PHASE}: {len(ALL_JOBS_PHASE)} jobs total | "
      f"cached {len(ALL_JOBS_PHASE) - len(PENDING)} | "
      f"pending {len(PENDING)}")

# ── Print already-completed phases ───────────────────────────────────────────
already_done = [p for p, s in PHASE_STATUS.items() if s == 'COMPLETE']
if already_done:
    print(f"  Previously completed phases: {already_done}")

# ═══════════════════════════════════════════════════════════════════════════
# BUDGETED EXECUTION LOOP — THIS PHASE ONLY
# ═══════════════════════════════════════════════════════════════════════════
budget_s      = TIME_BUDGET_MINUTES * 60.0
t_loop        = time.time()
done_this_run = 0
budget_hit    = False

for jid, fn in PENDING:
    if time.time() - t_loop > budget_s:
        budget_hit = True
        print(f"\n[BUDGET] {TIME_BUDGET_MINUTES} min reached after "
              f"{done_this_run} jobs. STOPPING CLEANLY — "
              f"re-run this cell to continue from cache.")
        break

    t0 = time.time()
    print(f"\n─── JOB {jid} "
          + "─" * max(0, 60 - len(jid)))
    try:
        r = fn()
        r = jclean(r)
        if not isinstance(r, dict):
            r = {'result': r}
    except Exception as e:
        r = {'status':    'ERROR',
             'error':     f'{type(e).__name__}: {e}',
             'traceback': traceback.format_exc()[-1500:]}
        print(f"  [ERROR] {jid}: {r['error']}")

    r['job_id'] = jid
    r['wall_s'] = round(time.time() - t0, 1)
    CACHE[jid]  = r
    save_cache()       # ← writes /kaggle/working/fme_cache/mega_checkpoint.json
    done_this_run += 1

    rem = len(PENDING) - done_this_run
    avg = (time.time() - t_loop) / max(done_this_run, 1)
    print("RESULT_JSON " + json.dumps(r, sort_keys=True, default=str))
    print(f"  [{jid}] done in {r['wall_s']:.0f}s | "
          f"pending remaining {rem} | "
          f"ETA≈{avg * rem / 60:.0f} min")

# ── Mark phase complete if all jobs finished ─────────────────────────────────
phase_finished = all(jid in CACHE for jid, _ in ALL_JOBS_PHASE)
if phase_finished:
    PHASE_STATUS[CURRENT_PHASE] = 'COMPLETE'
    save_phase_status()
    print(f"\n  ✓ Phase {CURRENT_PHASE} COMPLETE — "
          f"all {len(ALL_JOBS_PHASE)} jobs finished and cached.")
else:
    PHASE_STATUS[CURRENT_PHASE] = 'PARTIAL'
    save_phase_status()
    remaining = sum(1 for jid, _ in ALL_JOBS_PHASE if jid not in CACHE)
    print(f"\n  Phase {CURRENT_PHASE} PARTIAL — "
          f"{remaining} jobs still pending. "
          f"Re-run this cell to continue.")

# ═══════════════════════════════════════════════════════════════════════════
# RESULTS SUMMARY — prints ALL cached results across ALL phases
# ═══════════════════════════════════════════════════════════════════════════
def msd(vals):
    a = np.array([v for v in vals if v is not None], dtype=float)
    if len(a) == 0:
        return float('nan'), 0.0, 0
    return (float(a.mean()),
            (float(a.std(ddof=1)) if len(a) > 1 else 0.0),
            len(a))

def ci95(vals):
    m, s, n = msd(vals)
    return (float(st.t.ppf(0.975, n - 1) * s / np.sqrt(n))
            if n > 1 else 0.0)

def paired(v1, v2):
    v1 = np.array(v1, dtype=float); v2 = np.array(v2, dtype=float)
    d  = v1 - v2; n = len(d)
    if n < 2: return {'n': n}
    m  = float(d.mean())
    s  = float(d.std(ddof=1)) if n > 1 else 0.0
    ci = (float(st.t.ppf(0.975, n - 1) * s / np.sqrt(n))
          if s > 0 else 0.0)
    tp = (float(st.ttest_rel(v1, v2).pvalue)
          if s > 0 else float('nan'))
    try:
        wp = (float(st.wilcoxon(v1, v2).pvalue)
              if (n >= 6 and s > 0) else float('nan'))
    except Exception:
        wp = float('nan')
    return {'n': n, 'delta': m, 'sd': s,
            'ci_lo': m - ci, 'ci_hi': m + ci,
            't_p': tp, 'wilcoxon_p': wp}

def fmt_cmp(c):
    if c.get('n', 0) < 2:
        return f"n={c.get('n',0)} (insufficient paired data yet)"
    sig = ('n.s.' if (c['t_p'] != c['t_p'] or c['t_p'] >= 0.05)
           else 'SIGNIFICANT p<0.05')
    return (f"n={c['n']}  Δ={c['delta']:+6.2f} ± {c['sd']:4.2f} "
            f"(95%CI {c['ci_lo']:+.2f}…{c['ci_hi']:+.2f})  "
            f"t-p={c['t_p']:.4f}  "
            f"wilcoxon-p={c['wilcoxon_p']:.4f}  {sig}")

print("\n" + "═" * 78)
print("  FULL RESULTS — ALL CACHED JOBS ACROSS ALL PHASES")
print("═" * 78)
print("CACHE_JSON_BEGIN")
print(json.dumps(CACHE, indent=1, sort_keys=True, default=str))
print("CACHE_JSON_END")

AGG = {}

# ── Phase A ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 2 — PHASE A: multi-seed ablation (Tables 3-7)")
print("═" * 78)
aggA = {}
for block, cfgname in ORDER_A:
    rows = [CACHE.get(f"A|{s}|{block}|{cfgname}") for s in SEEDS_A]
    rows = [r for r in rows
            if r and r.get('status') not in ('ERROR', 'SKIPPED')]
    if not rows:
        print(f"  {block}/{cfgname}: no completed seeds yet")
        continue
    ua = [r['UA'] for r in rows]
    f1 = [r['F1'] for r in rows]
    rt = [r['RTF'] for r in rows]
    mU, sU, n = msd(ua)
    pub  = PUBLISHED_A.get(block, {}).get(cfgname)
    flag = ''
    if pub and not SMOKE:
        flag = (
            '  [published UA %.1f within range]' % pub[0]
            if min(ua) - 1e-9 <= pub[0] <= max(ua) + 1e-9
            else '  [published UA %.1f OUTSIDE range!]' % pub[0])
    print(f"\n  {block}/{cfgname}  (n={n})")
    print(f"    UA  = {mU:6.2f} ± {sU:4.2f}  "
          f"(95%CI ±{ci95(ua):4.2f}){flag}")
    print(f"    F1  = {msd(f1)[0]:6.2f} ± {msd(f1)[1]:4.2f}")
    print(f"    RTF = {msd(rt)[0]:.5f} ± {msd(rt)[1]:.5f}")
    print("    per-seed UA: " +
          ", ".join(f"{r['seed']}:{r['UA']:.1f}" for r in rows))
    print("    per-seed F1: " +
          ", ".join(f"{r['seed']}:{r['F1']:.1f}" for r in rows))
    aggA[f"{block}/{cfgname}"] = {
        'UA': ua, 'F1': f1, 'RTF': rt,
        'seeds': [r['seed'] for r in rows],
        'UA_mean': mU, 'UA_sd': sU}

cmpA = []
print("\n  Paired comparisons (UA):")
for b1, c1, b2, c2, label in PAIRS_A:
    k1, k2 = f"{b1}/{c1}", f"{b2}/{c2}"
    if k1 not in aggA or k2 not in aggA:
        print(f"  {label:<52s}: data incomplete")
        continue
    common = [s for s in aggA[k1]['seeds']
              if s in aggA[k2]['seeds']]
    v1 = [aggA[k1]['UA'][aggA[k1]['seeds'].index(s)]
          for s in common]
    v2 = [aggA[k2]['UA'][aggA[k2]['seeds'].index(s)]
          for s in common]
    c  = paired(v1, v2)
    print(f"  {label:<52s} {fmt_cmp(c)}")
    cmpA.append({'label': label, **c})
AGG['A'] = {'configs': aggA, 'comparisons': cmpA}

# ── Phase B ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 3 — PHASE B: Hurst ON (λ=1.0) vs OFF (λ=0)")
print("═" * 78)
bON  = [CACHE.get(f"B|lam1.0|{s}") for s in SEEDS_B]
bOFF = [CACHE.get(f"B|lam0.0|{s}") for s in SEEDS_B]
bON  = [r for r in bON
        if r and r.get('status') not in ('ERROR', 'SKIPPED')]
bOFF = [r for r in bOFF
        if r and r.get('status') not in ('ERROR', 'SKIPPED')]
aggB = {}
for tag, rows in (('ON λ=1.0', bON), ('OFF λ=0.0', bOFF)):
    if not rows:
        print(f"  {tag}: no completed seeds yet")
        continue
    print(f"  {tag} (n={len(rows)}): "
          f"UA {msd([r['UA'] for r in rows])[0]:.2f} ± "
          f"{msd([r['UA'] for r in rows])[1]:.2f} | "
          f"F1 {msd([r['F1'] for r in rows])[0]:.2f} ± "
          f"{msd([r['F1'] for r in rows])[1]:.2f} | "
          f"final H_mean "
          f"{msd([r['hurst_final'] for r in rows])[0]:.3f}")
    for r in rows:
        print(f"      seed {r['seed']}: "
              f"UA={r['UA']:.1f} F1={r['F1']:.1f} "
              f"H/epoch="
              f"{[round(x,3) for x in r['hurst_per_epoch']]}")
    aggB[tag] = {
        'UA': [r['UA'] for r in rows],
        'F1': [r['F1'] for r in rows],
        'hurst_final': [r['hurst_final'] for r in rows],
        'seeds': [r['seed'] for r in rows]}
if bON and bOFF:
    common = [s for s in aggB['ON λ=1.0']['seeds']
              if s in aggB['OFF λ=0.0']['seeds']]
    v1 = [aggB['ON λ=1.0']['UA']
          [aggB['ON λ=1.0']['seeds'].index(s)] for s in common]
    v2 = [aggB['OFF λ=0.0']['UA']
          [aggB['OFF λ=0.0']['seeds'].index(s)] for s in common]
    c  = paired(v1, v2)
    print(f"  R2-7 Hurst ON vs OFF (UA): {fmt_cmp(c)}")
    aggB['comparison_ON_minus_OFF'] = c
AGG['B'] = aggB

# ── Phase C ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 4 — PHASE C: Table-2 quick config, per seed")
print("═" * 78)
crows = [CACHE.get(f"C|{s}") for s in SEEDS_C]
crows = [r for r in crows
         if r and r.get('status') not in ('ERROR', 'SKIPPED')]
aggC  = {'per_seed': []}
for r in crows:
    tm    = r.get('test_metrics', {})
    eff   = r.get('efficiency', {})
    rtf10 = (eff.get('10s') or {}).get('rtf')
    line  = {'seed': r['seed'], 'test_metrics': tm,
             'rtf10s': rtf10, 'params': r.get('params')}
    aggC['per_seed'].append(line)
    print(f"  seed {r['seed']}: "
          f"{json.dumps(tm, default=str)}  "
          f"RTF(10s)={rtf10}  params={r.get('params')}")
if crows:
    def _numkey(d, *names):
        for n in names:
            if n in d:
                try: return float(d[n])
                except Exception: pass
        return None
    for label, names in (
            ('Acc',      ('test_acc','accuracy','acc','Acc')),
            ('UA',       ('test_ua','ua','UA','unweighted_acc')),
            ('F1_macro', ('test_f1_m','f1_macro','f1',
                          'macro_f1','F1'))):
        vals = [v for v in
                (_numkey(r.get('test_metrics', {}), *names)
                 for r in crows) if v is not None]
        if vals:
            pub_note = (
                "  [published: Acc/UA 80.00, F1(M) 77.43]"
                if label in ('Acc', 'F1_macro') and not SMOKE
                else "")
            print(f"  {label}: "
                  f"{msd(vals)[0]:.2f} ± {msd(vals)[1]:.2f} "
                  f"(n={len(vals)}){pub_note}")
            aggC[label] = {'vals': vals,
                           'mean': msd(vals)[0],
                           'sd':   msd(vals)[1]}
    r10 = [v for v in
           ((r.get('efficiency', {}).get('10s') or {}).get('rtf')
            for r in crows) if v is not None]
    if r10:
        pub_note = ("  [published 0.5806]" if not SMOKE else "")
        print(f"  RTF(10s, single-utterance): "
              f"{msd(r10)[0]:.4f} ± {msd(r10)[1]:.4f}{pub_note}")
        aggC['rtf10s'] = {'vals': r10,
                          'mean': msd(r10)[0],
                          'sd':   msd(r10)[1]}
AGG['C'] = aggC

# ── Phase D ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 5 — PHASE D: unified RTF (both protocols)")
print("═" * 78)
aggD = {}
for variant in ('quick3', 'abl4'):
    for protocol in ('single_utterance', 'batched'):
        r = CACHE.get(f"D|{variant}|{protocol}")
        if not r or r.get('status') in ('ERROR', 'SKIPPED'):
            continue
        aggD[f"{variant}|{protocol}"] = r
        if protocol == 'batched':
            print(f"  {variant:<7s} batched-throughput RTF = "
                  f"{r.get('rtf_batched')}")
        else:
            print(f"  {variant:<7s} single-utterance latency: "
                  f"{json.dumps(r.get('latency'), default=str)}")
AGG['D'] = aggD

# ── Phase E ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 6 — PHASE E: FME on REAL corpora (test-split metrics)")
print("═" * 78)
aggE = {}
for corpus in REAL_CORPORA:
    rows = [CACHE.get(f"E|{corpus}|{s}") for s in SEEDS_E]
    rows = [r for r in rows
            if r and r.get('status') not in ('ERROR', 'SKIPPED')]
    if not rows:
        print(f"  {corpus}: no completed seeds yet")
        continue
    print(f"\n  {corpus}  "
          f"({rows[0].get('corpus_info',{}).get('split_sizes')} "
          f"speakers="
          f"{rows[0].get('corpus_info',{}).get('n_actors')})")
    accs = [r['test']['acc']      for r in rows]
    uas  = [r['test']['UA']       for r in rows]
    f1s  = [r['test']['macroF1']  for r in rows]
    for r in rows:
        pc = {k: round(v,1) for k,v in r['test']['per_class_recall'].items()}
        print(f"    seed {r['seed']}: "
              f"Acc={r['test']['acc']:.2f} "
              f"UA={r['test']['UA']:.2f} "
              f"F1={r['test']['macroF1']:.2f}  "
              f"per-class={pc}  "
              f"RTF={r['rtf_batched']:.4f}")
    print(f"    MEAN: Acc {msd(accs)[0]:.2f}±{msd(accs)[1]:.2f} | "
          f"UA {msd(uas)[0]:.2f}±{msd(uas)[1]:.2f} | "
          f"F1 {msd(f1s)[0]:.2f}±{msd(f1s)[1]:.2f}")
    aggE[corpus] = {
        'acc': accs, 'UA': uas, 'F1': f1s,
        'confusion_last_seed': rows[-1]['test']['confusion'],
        'corpus_info': rows[0].get('corpus_info')}
AGG['E'] = aggE

# ── Phase F ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 7 — PHASE F: SSL baselines (fine-tuned) + RTF")
print("═" * 78)
aggF = {}
for corpus in REAL_CORPORA:
    for mkey in SSL_MODELS:
        rows = [CACHE.get(f"F|ft|{mkey}|{corpus}|{s}")
                for s in SEEDS_F]
        rows = [r for r in rows
                if r and r.get('status') not in ('ERROR','SKIPPED')]
        if not rows: continue
        accs = [r['test']['acc']     for r in rows]
        uas  = [r['test']['UA']      for r in rows]
        f1s  = [r['test']['macroF1'] for r in rows]
        print(f"  {mkey:<9s} {corpus:<8s} "
              f"Acc {msd(accs)[0]:6.2f}±{msd(accs)[1]:4.2f} | "
              f"UA {msd(uas)[0]:6.2f}±{msd(uas)[1]:4.2f} | "
              f"F1 {msd(f1s)[0]:6.2f}±{msd(f1s)[1]:4.2f}  "
              f"(params {rows[0]['params']/1e6:.0f}M)")
        aggF[f"{mkey}|{corpus}"] = {
            'acc': accs, 'UA': uas, 'F1': f1s}
    e2v = CACHE.get(f"F|e2v|{corpus}")
    if e2v:
        print(f"  emotion2vec {corpus}: "
              f"{json.dumps(e2v.get('test', e2v.get('status')), default=str)}")
        aggF[f"emotion2vec|{corpus}"] = e2v
    if corpus in aggE:
        for mkey in SSL_MODELS:
            k = f"{mkey}|{corpus}"
            if k in aggF and 'UA' in aggF[k]:
                d = paired(aggE[corpus]['UA'], aggF[k]['UA'])
                if d.get('n', 0) >= 2:
                    print(f"    ΔUA FME − {mkey} ({corpus}): "
                          f"{fmt_cmp(d)}")
                    aggF.setdefault(
                        'deltas', {})[f"FME-{mkey}|{corpus}"] = d
for mkey in SSL_MODELS:
    r = CACHE.get(f"F|rtf|{mkey}")
    if r and r.get('status') not in ('ERROR', 'SKIPPED'):
        print(f"  RTF {mkey}: "
              f"batched(8s)={r.get('rtf_batched_8s')}  "
              f"single="
              f"{json.dumps(r.get('latency_single_utterance'), default=str)}")
        aggF[f"rtf|{mkey}"] = r
AGG['F'] = aggF

# ── Phase G ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 8 — PHASE G: long-form 20s/60s × scales 1/4/5")
print("═" * 78)
aggG = {}
for dur in (20, 60):
    for ns in (1, 4, 5):
        rows = [CACHE.get(f"G|{dur}s|S{ns}|{s}") for s in SEEDS_G]
        rows = [r for r in rows
                if r and r.get('status') not in ('ERROR','SKIPPED')]
        if not rows: continue
        ua = [r['test']['UA'] for r in rows]
        print(f"  {dur}s S={ns} (n={len(rows)}): "
              f"test UA {msd(ua)[0]:6.2f}±{msd(ua)[1]:4.2f} | "
              f"val UA "
              f"{msd([r['UA_val'] for r in rows])[0]:6.2f} | "
              f"RTF "
              f"{msd([r['RTF'] for r in rows])[0]:.4f} | "
              f"per-seed UA(test): " +
              ", ".join(f"{r['seed']}:{r['test']['UA']:.1f}"
                        for r in rows))
        aggG[f"{dur}s|S{ns}"] = {
            'UA_test': ua,
            'RTF':    [r['RTF'] for r in rows],
            'seeds':  [r['seed'] for r in rows]}
AGG['G'] = aggG

# ── Phase H ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 9 — PHASE H: CTC lexical decoding (LibriSpeech-clean)")
print("═" * 78)
aggH = {}
prep = CACHE.get('H|prep')
if prep:
    print(f"  prep: "
          f"{json.dumps({k: v for k, v in prep.items() if k != 'traceback'}, default=str)}")
    aggH['prep'] = prep
for ep in range(1, CTC_EPOCHS + 1):
    r = CACHE.get(f"H|ep{ep}")
    if not r: continue
    if r.get('status') in ('ERROR', 'SKIPPED'):
        print(f"  epoch {ep}: {r.get('status')} — "
              f"{r.get('reason', r.get('error', ''))}")
    else:
        print(f"  epoch {ep}: WER={r['WER']:.1f}%  "
              f"CER={r['CER']:.1f}%  "
              f"loss={r['train_loss']:.3f}  "
              f"({r['time_min']:.0f} min, n={r['eval_n']})")
        for s in r.get('decode_samples', [])[:3]:
            print(f"      ref: {s['ref']}\n"
                  f"      hyp: {s['hyp']}")
    aggH[f'ep{ep}'] = r
AGG['H'] = aggH

# ── Machine-readable AGG ─────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 10 — AGG_JSON (machine-readable)")
print("═" * 78)
print("AGG_JSON_BEGIN")
print(json.dumps(AGG, indent=1, default=str))
print("AGG_JSON_END")

# ── Run status ───────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
print("  SECTION 11 — RUN STATUS")
print("═" * 78)

# Count across ALL phases (not just current)
all_phase_jobs = []
for ph in PHASES_ENABLED:
    try:
        all_phase_jobs.extend(PHASE_BUILDERS[ph]())
    except Exception:
        pass

finished_all = sum(1 for jid, _ in all_phase_jobs if jid in CACHE)
errors_all   = [jid for jid, _ in all_phase_jobs
                if CACHE.get(jid, {}).get('status') == 'ERROR']
skips_all    = [jid for jid, _ in all_phase_jobs
                if CACHE.get(jid, {}).get('status') == 'SKIPPED']

print(f"  CURRENT SESSION: phase={CURRENT_PHASE}  "
      f"jobs_run={done_this_run}  "
      f"phase_complete={phase_finished}")
print(f"  ALL PHASES: "
      f"finished {finished_all}/{len(all_phase_jobs)} | "
      f"errors: {len(errors_all)} {errors_all[:8]} | "
      f"skipped: {len(skips_all)} {skips_all[:8]}")

by_phase = defaultdict(lambda: [0, 0])
for jid, _ in all_phase_jobs:
    ph = jid.split('|')[0]
    by_phase[ph][1] += 1
    if jid in CACHE:
        by_phase[ph][0] += 1

print("  per-phase: " +
      "  ".join(f"{p}:{d}/{t}"
                for p, (d, t) in sorted(by_phase.items())))

completed_phases = [p for p, s in PHASE_STATUS.items()
                    if s == 'COMPLETE']
remaining_phases = [p for p in PHASES_ENABLED
                    if p not in completed_phases]

print(f"  completed phases: {completed_phases}")
print(f"  remaining phases: {remaining_phases}")

if finished_all == len(all_phase_jobs):
    print("\n  ✓✓✓ ALL PHASES COMPLETE — full experiment campaign finished.")
    print("      Complete record printed above "
          "(SECTION 1 CACHE_JSON + SECTION 10 AGG_JSON).")
else:
    if remaining_phases:
        print(f"\n  NEXT SESSION: set CURRENT_PHASE = "
              f"'{remaining_phases[0]}' in Cell 1, "
              f"then run Cell 1 → Cell 2.")
    else:
        print(f"\n  Some jobs still pending in current phase. "
              f"Re-run Cell 2 to continue.")

print("═" * 78)

  FME ALL-EXPERIMENTS — SEQUENTIAL SESSIONS
  device=cuda (Tesla T4)
  torch=2.10.0+cu128  SMOKE=False  budget=999999 min
  CURRENT_PHASE=B
  cache=/kaggle/working/fme_cache/mega_checkpoint.json
[resume] cache loaded: 4 finished jobs from previous sessions
[status] completed phases: ['D']
  [embedded AB] loaded in 0.0s
  [embedded QP] loaded in 0.0s
  [embedded H] loaded in 0.0s

  CELL 1 COMPLETE — Infrastructure loaded.
  Cache: 4 jobs from previous sessions.
  Phase status: {'D': 'COMPLETE'}
  CURRENT_PHASE = 'B'
  Now run Cell 2 to execute this phase.
  EXECUTING PHASE: B
  device=cuda  SMOKE=False
  cache has 4 jobs from previous sessions

  Phase B: 10 jobs total | cached 0 | pending 10
  Previously completed phases: ['D']

─── JOB B|lam1.0|42 ─────────────────────────────────────────────────

-------------------------------------------------------
  Creating HARD Emotion Data
-------------------------------------------------------
  [train] synthetic-hard: 2000 samples (neu=606,